# CS781 Capstone — Phase 7: Prior Case Retrieval (PCR) on 2×T4

**Kaggle settings:** GPU T4 ×2, Internet **on**, secret **`HF`** or **`HF_TOKEN`** (ticked for this notebook). **Add Input:**
the private Phase-1 model datasets (A, B2, C+D, B3 = `ctxext/best_newctx.pt`; only the ones your matrix uses); from a
second session on also earlier outputs of this notebook, so finished runs are reused (and can be KD teachers).

**Task plumbing:** bi-encoder on your Phase-1 models (Phase-4/5 chunked document encoder: chunks -> chunk pooling ->
attention / mean pooling over chunks -> optional projection), score = cosine or dot / temperature, InfoNCE over each
(query, relevant case) pair's candidates (BM25 hard or random negatives, optional in-batch negatives), gradient caching
so a step's documents share one loss. Selection and evaluation: dev Micro-F1@7 against `dev_candidates.json`.
Output: `prediction.json` = {query id: 7 candidate ids from `test_candidates.json`}. KD: your `kd_loss` over the
student's and the teacher's scores of the same candidate rows.

**Your parts (course rule 9):** `techniques/*.py` (Phase-5 files + IA³ from Phase 3), every value in `experiments.py`,
the fusion weights in Stage 7, and all analysis.

**ACCOUNT (Stage 3):** `1` / `2` = this account's share of `experiments.SPLIT`, `"all"` = everything here.
Outputs: `/kaggle/working/phase7/runs/<name>/`, zips in `/kaggle/working/submissions/`.

## Stage 0 — session, folders

In [ ]:
import os, sys, json, glob, time, hashlib, subprocess, shutil, inspect
SESSION_START = time.time()
WORK = "/kaggle/working/phase7"
for d in ("phase1_models", "techniques", "configs", "runs", "data", "cache"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

## Stage 1 — plumbing files (your Phase-1 model code unchanged; declare the rest)

In [ ]:
%%writefile /kaggle/working/phase7/a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/rope.py
# your rope.py (two lines fixed, marked FIX (Claude))
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/attention3.py
# your attention3.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/non_causal_self_attention.py
# your non_causal_self_attention.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/cross_attention.py
# your cross_attention.py (verbatim)
"""Cross Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x , memory):
        B, T, C = x.shape
        _,S,_ = memory.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/encoder.py
# your encoder.py (verbatim)
"""Encoder-Block of  Transformer LM -- Block now passes use_rope through to attention, and
EncoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from non_causal_self_attention import SelfAttention


@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class EncBlock(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = SelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class EncoderLM(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        #self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        #self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        #x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/decoder4.py
# your decoder4.py (verbatim)
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention3 import CausalSelfAttention
from cross_attention import CrossAttention


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class DecBlock(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.crattn = CrossAttention(cfg.d_model , cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,
        cfg.dropout,use_rope=False,attention_scale=cfg.attention_scale)

        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x , memory):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.crattn(y3,memory)+y2
            y5 = self.ln3(y4)
            y6 = self.ffn(y5)+y4
            output=y6
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.crattn(y2,memory)
            y4 = self.ln2(y3+y2)
            y5 = self.ffn(y4)
            y6 = self.ln3(y5+y4)
            output=y6
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor , memory:torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,memory)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/enc_dec.py
# your enc_dec.py (verbatim)
from dataclasses import dataclass 
import torch  
import torch.nn as nn  
from encoder import EncBlock, sinusoidal_positional_encoding  
from decoder4 import DecBlock

@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


class EncDecLM(nn.Module):
    def __init__(self,cfg1:EncLMConfig,cfg2:DecLMConfig):
        super().__init__()
        # Encoder Block Initializations 
        self.cfg1 = cfg1
        self.tok_emb1 = nn.Embedding(cfg1.vocab_size, cfg1.d_model)
        if cfg1.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg1.max_seq_len, cfg1.d_model)
            self.register_buffer("pos_emb_enc", pe, persistent=False)
   
        self.encblocks = nn.ModuleList([EncBlock(cfg1) for _ in range(cfg1.n_layers)])
        self.ln_f_enc = nn.LayerNorm(cfg1.d_model)
        # Decoder Block Initializations
        self.cfg2 = cfg2
        self.tok_emb2 = nn.Embedding(cfg2.vocab_size, cfg2.d_model)
        if cfg2.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg2.max_seq_len, cfg2.d_model)
            self.register_buffer("pos_emb_dec", pe, persistent=False)
       
        self.decblocks = nn.ModuleList([DecBlock(cfg2) for _ in range(cfg2.n_layers)])
        self.ln_f_dec = nn.LayerNorm(cfg2.d_model)
        self.head = nn.Linear(cfg2.d_model, cfg2.vocab_size, bias=False)
        self.head.weight = self.tok_emb2.weight
    def encode(self,src:torch.Tensor)->torch.Tensor:
        # passing it through encoder 
        B,S = src.shape
        
        x_enc = self.tok_emb1(src)
        if(self.cfg1.pos_emb_type == "sinusoidal"):
            x_enc=x_enc+self.pos_emb_enc[:S]
        for block in self.encblocks:
            x_enc = block(x_enc)
        x_enc = self.ln_f_enc(x_enc)
        memory = x_enc
        return memory

    def decode(self,tgt:torch.Tensor,memory:torch.Tensor)->torch.Tensor:
        B,T = tgt.shape 
        x_dec = self.tok_emb2(tgt)
        if(self.cfg2.pos_emb_type == "sinusoidal"):
            x_dec=x_dec+self.pos_emb_dec[:T]
        for block in self.decblocks:
            x_dec = block(x_dec,memory)
        x_dec = self.ln_f_dec(x_dec)
        x_dec = self.head(x_dec)
        return x_dec


    def forward(self ,src:torch.Tensor , tgt:torch.Tensor)->torch.Tensor:
        return self.decode(tgt,self.encode(src))
        
         
        






        
    
        

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/prefix_attention.py
# your prefix_attention.py (verbatim)
""" Prefix Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
     

    def forward(self, x , p):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale

        mask1 = self.causal_mask[:T,:T].clone()
        mask1[:p,:p]=1

        attention_scores = attention_scores.masked_fill(
            ~mask1,float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/prefix_lm.py
# your prefix_lm.py (verbatim)
"""Prefix LM - Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2). Different Architecture as mask changes the way information flows"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from prefix_attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False
    prefix:int = 128


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x,p):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1,p)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x,p)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,self.cfg.prefix)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/fastattn_ab.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)
    n_prefix = 0                                            # prefix tuning is only wired for MHA / GQA

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        k, v, n_prefix = _add_prefix(self, k, v, B, cache)
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if n_prefix:                                            # prefix tuning: every query also sees the P prefix keys
        mask = _causal_with_prefix(T, n_prefix, x.device)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p, scale=self.attention_scale)
    else:
        y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


# ---------------------------------------------------------------------------
# prefix-tuning hook (Phase 3 plumbing): an attention module may carry `prefix_kv`, a callable
# prefix_kv(batch_size) -> (pk, pv), each (B, n_kv_heads, P, d_head). Those P key/value vectors are prepended to the
# keys/values (no RoPE is applied to them) and every query may attend to them. Without `prefix_kv`: unchanged.
# ---------------------------------------------------------------------------

def _add_prefix(attn, k, v, B, cache):
    fn = getattr(attn, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    assert cache is None, "prefix tuning is not supported together with the generation KV cache"
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _causal_with_prefix(T, P, device):
    m = torch.ones(T, P + T, dtype=torch.bool, device=device)
    m[:, P:] = torch.ones(T, T, dtype=torch.bool, device=device).tril()
    return m


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


@torch.inference_mode()
def generate_beam(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens, batch_size,
                  beam_size, device, amp_dtype, no_repeat_ngram=0, ngram_scope="output", length_penalty=1.0,
                  log_every=10):
    """Beam search with the same word budget / token constraints as generate_words.
    Beams are ranked by sum(log p) / length**length_penalty (1.0 = mean log-prob per token). A beam that has
    produced `target_words` words is frozen (score and length fixed) and keeps competing as a finished
    hypothesis. Optional n-gram blocking per beam (same rule as generate_words). Returns list[str] in input order."""
    assert ngram_scope in ("output", "all"), ngram_scope
    model.eval()
    W = int(beam_size)
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    n_decodable = len(tokenizer.token_bytes)
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    per_batch = max(1, batch_size // W)
    results = [None] * len(prompts)
    neg_inf = -float("inf")

    for b, start in enumerate(range(0, len(order), per_batch)):
        idxs = order[start:start + per_batch]
        P = len(idxs)
        R = P * W
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        prompt_ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        ids = prompt_ids.repeat_interleave(W, 0)                                   # (R, L): row = p * W + beam
        words = torch.zeros(R, dtype=torch.int32, device=device)
        in_word = torch.zeros(R, dtype=torch.bool, device=device)
        done = torch.zeros(R, dtype=torch.bool, device=device)
        cum = torch.zeros(R, device=device)
        length = torch.zeros(R, device=device)
        gen = torch.zeros(R, 0, dtype=torch.long, device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, R, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            for t in range(max_new_tokens):
                lp = torch.nan_to_num(logits.float(), nan=-1e4)
                lp[:, :tokenizer.n_special] = neg_inf
                lp[~in_word, tokenizer.eow_id] = neg_inf
                lp[:, n_decodable:] = neg_inf
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = gen if ngram_scope == "output" else torch.cat([ids, gen], 1)
                    ban = ngram_ban(seq, no_repeat_ngram, lp.size(1))
                    ban &= ~(ban | torch.isinf(lp)).all(-1, keepdim=True)
                    lp = lp.masked_fill(ban, neg_inf)
                lp = lp.log_softmax(-1)
                V = lp.size(1)
                # candidate scores (length-normalised); finished beams keep their score via one frozen continuation
                cand = (cum[:, None] + lp) / (length[:, None] + 1).pow(length_penalty)
                frozen_tok = lp.argmax(-1)
                frozen = torch.full_like(cand, neg_inf)
                frozen.scatter_(1, frozen_tok[:, None], (cum / length.clamp(min=1).pow(length_penalty))[:, None])
                cand = torch.where(done[:, None], frozen, cand)
                if t == 0:                                                         # identical beams: expand beam 0 only
                    cand.view(P, W, V)[:, 1:] = neg_inf
                top_s, top_i = cand.view(P, W * V).topk(W, dim=-1)                 # (P, W)
                parent = (top_i // V) + torch.arange(P, device=device)[:, None] * W
                parent, tok = parent.reshape(-1), (top_i % V).reshape(-1)
                was_done = done[parent]
                # reorder beam state + KV cache to the chosen parents
                for c in model._fast_state["caches"]:
                    pos = c["pos"]
                    c["k"][:, :, :pos] = c["k"][parent, :, :pos]
                    c["v"][:, :, :pos] = c["v"][parent, :, :pos]
                words, in_word, done = words[parent], in_word[parent], done[parent].clone()
                cum = torch.where(was_done, cum[parent], cum[parent] + lp[parent, tok])
                length = torch.where(was_done, length[parent], length[parent] + 1)
                gen = torch.cat([gen[parent], tok[:, None]], 1)
                open_ = in_word | has_bytes[tok]
                words = words + (eow[tok] & open_ & ~was_done).int()
                in_word = open_ & ~eow[tok]
                done = was_done | (words >= target_words)
                if bool(done.all()):
                    break
                logits = step(model, tok[:, None])[:, -1]
        score = (cum / length.clamp(min=1).pow(length_penalty)).view(P, W)
        score = torch.where(done.view(P, W), score, score - 1e6)                     # prefer finished beams
        best = score.argmax(-1) + torch.arange(P, device=device) * W
        rows = gen[best].tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(rows[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  beam generated {min(start + per_batch, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase7/phase1_models/fastattn_cd.py
# Plumbing: fused SDPA attention + KV caches + generation around your #3 / #4 modules
"""Efficiency plumbing around the hand-written encoder-decoder (#3) and prefix-LM (#4) (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay exactly as defined in
your files. `patch_model` swaps each attention module's forward for one that computes the *same* function with
F.scaled_dot_product_attention (memory-efficient fused kernel on T4), and adds optional caches for generation:

  encoder SelfAttention (non_causal_self_attention.py)  -> bidirectional, no cache (the encoder runs once per prompt)
  decoder CausalSelfAttention (attention3.py)           -> causal, self-attention KV cache
  CrossAttention (cross_attention.py)                   -> K/V of the encoder memory computed once per prompt, cached
  prefix CausalSelfAttention (prefix_attention.py)      -> mask = causal OR [:p, :p] block, KV cache for decoding

`check_equivalence_*` (run in the notebook before training) verify the patched model against your original forward,
and cached step-by-step decoding against a full forward pass.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forwards
# ---------------------------------------------------------------------------

def _dropout_p(self):
    return self.attn_dropout.p if self.training else 0.0


def _project_self(self, x, start):
    """q, k, v of a self-attention module, RoPE at absolute positions start..start+T (as in your code)."""
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    if getattr(self, "use_rope", False):
        cos = self.rope_cos[start:start + T]
        sin = self.rope_sin[start:start + T]
        q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
        k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
    return q, k, v


def _use_cache(self, k, v, start):
    cache = getattr(self, "_kv_cache", None)
    if cache is None:
        return k, v
    T = k.shape[2]
    cache["k"][:, :, start:start + T] = k
    cache["v"][:, :, start:start + T] = v
    return cache["k"][:, :, :start + T], cache["v"][:, :, :start + T]


def _expand(self, k, v):
    if self.n_rep > 1:
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
    return k, v


def _finish(self, y):
    B, H, T, D = y.shape
    return self.resid_dropout(self.out_proj(y.transpose(1, 2).reshape(B, T, H * D)))


def _sdpa(self, q, k, v, attn_mask=None, is_causal=False):
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal,
                                          dropout_p=_dropout_p(self), scale=self.attention_scale)


def _start(self):
    cache = getattr(self, "_kv_cache", None)
    return cache["pos"] if cache is not None else 0


def _prefix_hook(self, k, v, B):
    """prefix-tuning hook (Phase 3 plumbing): optional self.prefix_kv(B) -> (pk, pv), each (B, n_kv_heads, P, d_head),
    prepended to keys/values (no RoPE on them). Returns (k, v, P)."""
    fn = getattr(self, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    assert getattr(self, "_kv_cache", None) is None, "prefix tuning is not supported with the generation KV cache"
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _with_prefix_cols(mask_TT, P):
    """(T, T) bool mask -> (T, P + T): the P prefix columns are visible to every query."""
    T = mask_TT.shape[0]
    return torch.cat([torch.ones(T, P, dtype=torch.bool, device=mask_TT.device), mask_TT], dim=1)


def _fwd_bidirectional(self, x):
    """encoder SelfAttention: no mask."""
    q, k, v = _project_self(self, x, 0)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


def _fwd_causal(self, x):
    """decoder CausalSelfAttention: causal; one-token steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if T > 1 and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if P:
        mask = _with_prefix_cols(torch.ones(T, T, dtype=torch.bool, device=x.device).tril(), P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    return _finish(self, _sdpa(self, q, k, v, is_causal=T > 1))


def _prefix_mask(self, T, p, device):
    key = (T, int(p), str(device))
    masks = self.__dict__.setdefault("_mask_cache", {})
    if key not in masks:
        m = torch.ones(T, T, dtype=torch.bool, device=device).tril()
        m[:p, :p] = True                       # same rule as your prefix_attention.py
        masks[key] = m
    return masks[key]


def _fwd_prefix(self, x, p):
    """prefix attention: causal OR [:p, :p]; one-token decoding steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if start == 0:
        mask = _prefix_mask(self, T, p, x.device)
        if P:
            mask = _with_prefix_cols(mask, P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    if T == 1:
        return _finish(self, _sdpa(self, q, k, v))
    raise ValueError("multi-token steps are only supported from an empty cache")


def _fwd_cross(self, x, memory):
    """CrossAttention: q from the decoder state, k/v from the encoder memory (cached once per prompt)."""
    assert not getattr(self, "use_rope", False), "RoPE in cross-attention is not supported (your design: use_rope=False)"
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    cache = getattr(self, "_kv_cache", None)
    if cache is not None and "k" in cache:
        k, v = cache["k"], cache["v"]
    else:
        S = memory.shape[1]
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        if cache is not None:
            cache["k"], cache["v"] = k, v
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


_FORWARDS = {
    ("non_causal_self_attention", "SelfAttention"): _fwd_bidirectional,
    ("attention3", "CausalSelfAttention"): _fwd_causal,
    ("cross_attention", "CrossAttention"): _fwd_cross,
    ("prefix_attention", "CausalSelfAttention"): _fwd_prefix,
}


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    n = 0
    for mod in model.modules():
        fwd = _FORWARDS.get((type(mod).__module__, type(mod).__name__))
        if fwd is None:
            continue
        mod._kv_cache = None
        if getattr(mod, "use_rope", False):
            mod._apply_rope = sys.modules[type(mod).__module__].apply_rope   # your own apply_rope
        mod.forward = types.MethodType(fwd, mod)
        n += 1
    assert n > 0, "no attention module recognised; did a module/class name change?"
    return model


# ---------------------------------------------------------------------------
# generation sessions: prefill once, then one token per step
# ---------------------------------------------------------------------------

def _alloc_self_cache(attn, B, max_len, dtype, device):
    shape = (B, attn.n_kv_heads, max_len, attn.d_head)
    attn._kv_cache = {"pos": 0, "k": torch.empty(shape, device=device, dtype=dtype),
                      "v": torch.empty(shape, device=device, dtype=dtype)}
    return attn._kv_cache


@contextmanager
def prefix_session(model, ids, max_new_tokens, dtype):
    """#4: prefill the prompt (mask uses the model's own cfg.prefix), then decode with the KV cache."""
    B, L = ids.shape
    dev = ids.device
    caches = [_alloc_self_cache(b.attn, B, L + max_new_tokens, dtype, dev) for b in model.blocks]
    full_pe = getattr(model, "pos_emb", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb = full_pe[pos:]     # DecoderLM adds pos_emb[:T]; shift it to rows pos..pos+T
        logits = model(tok)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    try:
        yield run(ids), run
    finally:
        for b in model.blocks:
            b.attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe


@contextmanager
def encdec_session(model, ids, max_new_tokens, dtype, start_id=None):
    """#3: encode the prompt once, cache every decoder layer's cross-attention K/V, decode token by token.
    start_id=None -> the decoder starts from the last source token, else from start_id (e.g. <s>)."""
    B = ids.shape[0]
    dev = ids.device
    memory = model.encode(ids)
    caches = [_alloc_self_cache(b.attn, B, max_new_tokens + 1, dtype, dev) for b in model.decblocks]
    for b in model.decblocks:
        b.crattn._kv_cache = {}
    full_pe = getattr(model, "pos_emb_dec", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb_dec = full_pe[pos:]
        logits = model.decode(tok, memory)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    first = ids[:, -1:] if start_id is None else torch.full((B, 1), start_id, device=dev, dtype=ids.dtype)
    try:
        yield run(first), run
    finally:
        for b in model.decblocks:
            b.attn._kv_cache = None
            b.crattn._kv_cache = None
        if full_pe is not None:
            model.pos_emb_dec = full_pe


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, kind, max_prompt, target_words, max_new_tokens, batch_size,
                   temperature, top_k, top_p, repetition_penalty, device, amp_dtype, start_id=None, log_every=10):
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    kind = "prefix" (#4) or "encdec" (#3). Prompts are bucketed by token length so a batch needs no padding: each
    prompt keeps its last L tokens, L = shortest prompt in the batch (capped at max_prompt)."""
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompts)
    cache_dtype = amp_dtype if device.type == "cuda" else torch.float32

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        session = (prefix_session(model, ids, max_new_tokens, cache_dtype) if kind == "prefix"
                   else encdec_session(model, ids, max_new_tokens, cache_dtype, start_id))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), session as (logits, step):
            history = ids
            for t in range(max_new_tokens):
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")       # a bare </w> after a finished word
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()) or t == max_new_tokens - 1:
                    break
                if repetition_penalty != 1.0:
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(nxt[:, None])
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence_encdec(make_model, vocab_size, src_len=20, tgt_len=12, rtol=1e-4):
    """Your forward vs patched forward vs the generation path (encdec_session: encode once, cached cross-attention
    K/V, one decoder token per step). Returns relative max errors (fp32, CPU)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    src = torch.randint(0, vocab_size, (2, src_len))
    tgt = torch.randint(0, vocab_size, (2, tgt_len))
    tgt[:, 0] = src[:, -1]                    # session with start_id=None starts from the last source token
    a = ref(src, tgt)
    scale = a.abs().max().item()
    err_full = (a - fast(src, tgt)).abs().max().item() / scale
    with encdec_session(fast, src, tgt_len, torch.float32, start_id=None) as (first, step):
        outs = [first[:, None]]
        for t in range(1, tgt_len):
            outs.append(step(tgt[:, t:t + 1])[:, None])
    err_cache = (a - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache


@torch.no_grad()
def check_equivalence_prefix(make_model, vocab_size, prefix, seq_len=None, rtol=1e-4):
    """Your forward vs patched forward vs (prefill >= prefix tokens, then cached one-token steps)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    seq_len = seq_len or prefix + 12
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    n_new = seq_len - prefix
    with prefix_session(fast, idx[:, :prefix], n_new, torch.float32) as (first, step):
        outs = [first[:, None]]
        for t in range(prefix, seq_len - 1):
            outs.append(step(idx[:, t:t + 1])[:, None])
    err_cache = (a[:, prefix - 1:seq_len - 1] - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase7/fetch_data.py
#!/usr/bin/env python3
"""Download Phase_7_PCR/ from the gated course dataset (plumbing; declare it).

Token lookup order: Kaggle secret "HF" -> HF_TOKEN environment variable -> hidden prompt.
The token is used for these requests only: never printed, never written to disk.

  python fetch_data.py [--out DIR]          # -> DIR/Phase_7_PCR/{Train,Dev,Test}/...
"""
import argparse
import getpass
import json
import os
import sys
import urllib.error
import urllib.request

REPO = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
FILES = ["Phase_7_PCR/Train/train.json", "Phase_7_PCR/Train/train_candidates.json",
         "Phase_7_PCR/Dev/dev.json", "Phase_7_PCR/Dev/dev_candidates.json",
         "Phase_7_PCR/Test/test.json", "Phase_7_PCR/Test/test_candidates.json"]


def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if not sys.stdin.isatty():
        raise SystemExit("No Hugging Face token: on Kaggle add a secret with the LABEL 'HF' (Add-ons -> Secrets) and tick "
                         "it for this notebook; locally set HF_TOKEN or run this script in a terminal.")
    return getpass.getpass("Hugging Face token (input hidden): ").strip()


def get(url, token):
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            return r.read()
    except urllib.error.HTTPError as e:
        hint = "check the token / dataset access" if e.code in (401, 403) else e.reason
        raise SystemExit(f"download failed for {url.split('/main/')[-1]}: HTTP {e.code} ({hint})")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "data"))
    args = ap.parse_args()
    token = get_token()
    try:
        for rel in FILES:
            dst = os.path.join(args.out, rel)
            if os.path.exists(dst):
                print("exists:", dst); continue
            data = get(REPO + rel, token)
            json.loads(data)                                   # validate before saving
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst + ".tmp", "wb") as f:
                f.write(data)
            os.replace(dst + ".tmp", dst)
            print(f"saved {dst} ({len(data) / 1e6:.1f} MB)")
    finally:
        del token


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase7/pcr_data.py
"""PCR data: load queries + candidate pools, ByteBPE encoding (cached), chunk selection, BM25, Micro-F1@k
(plumbing; declare it).

Files (spec 13.3): Phase_7_PCR/{Train/train.json + train_candidates.json, Dev/dev.json + dev_candidates.json,
Test/test.json + test_candidates.json}. Each split's queries are ranked ONLY against that split's own pool.
The loader accepts a list of records or a dict keyed by case id; the id field may be called id / case_id / query_id /
candidate_id / doc_id; the text field (text / document / sentences / content) may be a list of sentences (joined with
one space, like the whitespace-normalised Phase-1 data) or a string. Relevance: "relevant_candidates" (spec), also
accepted: relevant / candidates / labels. Test relevance is never read (load_split drops it).

Chunking: same functions as Phases 4-5 (even | headtail | overlap windows of L tokens, at most k per document).
Metric (spec 13.4): Micro-P/R/F1@7 = hits summed over queries / (7 x queries) and / (gold relevant summed over
queries); the gold set is complete (never truncated), so recall counts relevant cases that are not in the top 7.
"""
import hashlib
import json
import math
import os
import re
from collections import Counter

import numpy as np

SPLITS = {"train": ("Train", "train.json", "train_candidates.json"),
          "dev": ("Dev", "dev.json", "dev_candidates.json"),
          "test": ("Test", "test.json", "test_candidates.json")}
ID_KEYS = ("id", "case_id", "query_id", "candidate_id", "doc_id")
TEXT_KEYS = ("text", "document", "sentences", "content")
REL_KEYS = ("relevant_candidates", "relevant", "candidates", "labels")
TOP_K = 7


# ---------------------------------------------------------------------------
# loading
# ---------------------------------------------------------------------------

def _first(d, keys):
    for k in keys:
        if k in d:
            return k
    return None


def _text(v):
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, list):
        return " ".join(str(s).strip() for s in v if str(s).strip())
    raise TypeError(f"unexpected text type {type(v).__name__}")


def _records(obj):
    """-> list of (id, record dict) from a list of records or an {id: record | text} dict."""
    if isinstance(obj, dict):
        for wrap in ("data", "queries", "candidates", "cases"):        # {"queries": [...]} style wrappers
            if wrap in obj and isinstance(obj[wrap], (list, dict)) and len(obj) <= 3:
                return _records(obj[wrap])
        out = []
        for k, v in obj.items():
            out.append((str(k), v if isinstance(v, dict) else {"text": v}))
        return out
    assert isinstance(obj, list), f"unexpected top-level JSON type {type(obj).__name__}"
    out = []
    for r in obj:
        k = _first(r, ID_KEYS)
        assert k, f"no id field in record with keys {sorted(r)[:10]}"
        out.append((str(r[k]), r))
    return out


def _docs(obj, with_rel):
    docs = []
    for i, r in _records(obj):
        tk = _first(r, TEXT_KEYS)
        assert tk, f"record {i}: no text field (keys {sorted(r)[:10]})"
        d = {"id": i, "text": _text(r[tk])}
        if with_rel:
            rk = _first(r, REL_KEYS)
            d["relevant"] = [str(x) for x in r[rk]] if rk and r[rk] is not None else None
        docs.append(d)
    ids = [d["id"] for d in docs]
    assert len(set(ids)) == len(ids), f"{len(ids) - len(set(ids))} duplicate ids"
    return docs


def load_split(data_dir, split):
    """-> (queries [{"id", "text", "relevant" (None for test)}], pool [{"id", "text"}])."""
    sub, qf, cf = SPLITS[split]
    queries = _docs(json.load(open(os.path.join(data_dir, sub, qf), encoding="utf-8")), with_rel=True)
    pool = _docs(json.load(open(os.path.join(data_dir, sub, cf), encoding="utf-8")), with_rel=False)
    if split == "test":
        for q in queries:
            q["relevant"] = None                       # test labels are private: never read
    else:
        assert all(q["relevant"] is not None for q in queries), f"{split}: queries without relevant_candidates"
    return queries, pool


def schema_report(data_dir):
    """first-record keys / types of every file + basic counts (printed by the notebook before anything else)."""
    rep = {}
    for split, (sub, qf, cf) in SPLITS.items():
        for f in (qf, cf):
            obj = json.load(open(os.path.join(data_dir, sub, f), encoding="utf-8"))
            recs = _records(obj)
            i, r = recs[0]
            rep[f] = {"top_level": type(obj).__name__, "n": len(recs), "first_id": i,
                      "keys": {k: type(v).__name__ for k, v in list(r.items())[:8]}}
    return rep


def split_stats(queries, pool):
    pid = {d["id"] for d in pool}
    st = {"queries": len(queries), "pool": len(pool),
          "pool_words_median": int(np.median([len(d["text"].split()) for d in pool])),
          "query_words_median": int(np.median([len(d["text"].split()) for d in queries])),
          "queries_in_own_pool": sum(q["id"] in pid for q in queries)}
    if queries and queries[0]["relevant"] is not None:
        rel = [len(q["relevant"]) for q in queries]
        st.update(rel_mean=round(float(np.mean(rel)), 2), rel_median=float(np.median(rel)), rel_max=max(rel),
                  rel_min=min(rel), rel_missing_from_pool=sum(c not in pid for q in queries for c in q["relevant"]),
                  rel_total=sum(rel))
    return st


def norm_hash(text):
    return hashlib.md5(" ".join(text.split()).encode("utf-8")).hexdigest()


# ---------------------------------------------------------------------------
# tokens + chunks (unchanged from Phase 4 / 5)
# ---------------------------------------------------------------------------

def encode_docs(tok, docs, cache_path=None):
    """-> list of np.int32 token arrays (one per doc, same order). Cached by document hashes."""
    key = hashlib.md5(("|".join(d["id"] + ":" + norm_hash(d["text"]) for d in docs)).encode()).hexdigest()
    if cache_path and os.path.exists(cache_path):
        with np.load(cache_path) as z:
            if str(z["key"]) == key:
                ids, off = z["ids"], z["offsets"]
                return [ids[off[i]:off[i + 1]] for i in range(len(docs))]
    out = [np.asarray(tok.encode(d["text"]), dtype=np.int32) for d in docs]
    assert all(len(a) > 0 for a in out), "a document encodes to zero tokens"
    if cache_path:
        offsets = np.concatenate([[0], np.cumsum([len(a) for a in out])]).astype(np.int64)
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        np.savez(cache_path + ".tmp.npz", ids=np.concatenate(out), offsets=offsets, key=key)
        os.replace(cache_path + ".tmp.npz", cache_path)
    return out


def _spread(n_windows, k):
    if n_windows <= k:
        return list(range(n_windows))
    idx = sorted({int(round(x)) for x in np.linspace(0, n_windows - 1, k)})
    assert len(idx) == k, (n_windows, k, idx)
    return idx


def chunk_spans(n, L, strategy, k, stride=None):
    """-> list of (start, end) token spans for a document of n tokens, in text order, no duplicates."""
    assert n > 0 and L > 0 and k >= 1
    if n <= L:
        return [(0, n)]
    if strategy in ("even", "headtail"):
        W = math.ceil(n / L)
        grid = [(i * L, min(n, (i + 1) * L)) for i in range(W)]
        if W <= k:
            return grid
        if strategy == "even":
            return [grid[i] for i in _spread(W, k)]
        h, t = math.ceil(k / 2), k // 2
        head = [(i * L, (i + 1) * L) for i in range(h)]
        tail = [(n - (j + 1) * L, n - j * L) for j in range(t)][::-1]
        return head + tail
    if strategy == "overlap":
        assert stride and 0 < stride <= L, "overlap needs 0 < stride <= L"
        starts = list(range(0, n - L + 1, stride))
        if starts[-1] != n - L:
            starts.append(n - L)
        wins = [(s, s + L) for s in starts]
        return [wins[i] for i in _spread(len(wins), k)]
    raise ValueError(f"unknown chunking strategy {strategy!r}: even | headtail | overlap")


def build_items(docs, token_arrays, L, strategy, k, stride=None):
    """-> list of {"id", "chunks": [np.int32 arrays], "n_tokens"}"""
    items = []
    for d, a in zip(docs, token_arrays):
        spans = chunk_spans(len(a), L, strategy, k, stride)
        chunks = [a[s:e] for s, e in spans]
        items.append({"id": d["id"], "chunks": chunks, "n_tokens": int(len(a))})
    return items


def coverage(items):
    return sum(sum(len(c) for c in it["chunks"]) for it in items) / max(1, sum(it["n_tokens"] for it in items))


# ---------------------------------------------------------------------------
# BM25 (lexical; used for hard negatives, as a local reference score and for optional score fusion)
# ---------------------------------------------------------------------------

WORD = re.compile(r"[a-z0-9]+")


def words(text):
    return [w for w in WORD.findall(text.lower()) if len(w) > 1]


class BM25:
    """Okapi BM25 over lower-cased words (length >= 2). score(q, d) = sum over the query's DISTINCT words t of
    idf(t) * tf(t,d) (k1+1) / (tf(t,d) + k1 (1 - b + b |d| / avgdl)), idf = ln(1 + (N - df + 0.5) / (df + 0.5))."""

    def __init__(self, docs, k1=1.2, b=0.75):
        self.ids = [d["id"] for d in docs]
        tfs = [Counter(words(d["text"])) for d in docs]
        dl = np.array([sum(c.values()) for c in tfs], dtype=np.float64)
        avg = dl.mean() if len(dl) else 1.0
        df = Counter(t for c in tfs for t in c)
        N = len(docs)
        post = {}
        for j, c in enumerate(tfs):
            norm = k1 * (1 - b + b * dl[j] / avg)
            for t, f in c.items():
                post.setdefault(t, ([], []))
                post[t][0].append(j)
                post[t][1].append(f * (k1 + 1) / (f + norm))
        self.post = {t: (np.asarray(js, dtype=np.int32),
                         np.asarray(ws, dtype=np.float32) * np.float32(math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5))))
                     for t, (js, ws) in post.items()}
        self.N = N

    def scores(self, text):
        s = np.zeros(self.N, dtype=np.float32)
        for t in set(words(text)):
            p = self.post.get(t)
            if p is not None:
                np.add.at(s, p[0], p[1])
        return s

    def score_matrix(self, queries):
        return np.stack([self.scores(q["text"]) for q in queries]) if queries else np.zeros((0, self.N), np.float32)


# ---------------------------------------------------------------------------
# ranking + metrics
# ---------------------------------------------------------------------------

def rank(S, qids, cids, k, exclude_self=True):
    """S (n_queries, n_pool) scores -> {qid: top-k candidate ids, best first}. A query's own id is never returned."""
    out = {}
    cidx = {c: j for j, c in enumerate(cids)}
    for i, q in enumerate(qids):
        row = np.asarray(S[i], dtype=np.float64).copy()
        if exclude_self and q in cidx:
            row[cidx[q]] = -np.inf
        top = np.argsort(-row, kind="stable")[:k]
        out[q] = [cids[j] for j in top]
    return out


def prf_at_k(ranked, gold, k=TOP_K):
    """micro precision / recall / F1 @k. ranked {qid: [ids]}, gold {qid: [ids]} (complete relevance sets)."""
    hits = sum(len(set(ranked[q][:k]) & set(g)) for q, g in gold.items())
    n_ret = k * len(gold)
    n_rel = sum(len(set(g)) for g in gold.values())
    p = hits / n_ret if n_ret else 0.0
    r = hits / n_rel if n_rel else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"micro_p": round(p, 4), "micro_r": round(r, 4), "micro_f1": round(f, 4), "hits": hits, "k": k}


def report(S, qids, cids, gold, ks=(1, 3, 5, 7, 10, 20)):
    """Micro-F1@7 (the metric) + P/R/F1 at other cut-offs and recall@100 for analysis."""
    deep = rank(S, qids, cids, max(max(ks), 100))
    rep = {"at7": prf_at_k(deep, gold, TOP_K)}
    rep["curve"] = {str(k): prf_at_k(deep, gold, k)["micro_f1"] for k in ks}
    rep["recall@100"] = prf_at_k(deep, gold, 100)["micro_r"]
    return rep


def write_prediction(S, qids, cids, path):
    """prediction.json (spec 13.3.5): {query id: [7 unique candidate ids from this split's pool, best first]}."""
    pred = rank(S, qids, cids, TOP_K)
    pool = set(cids)
    assert len(pred) == len(qids) and all(len(v) == TOP_K and len(set(v)) == TOP_K and set(v) <= pool
                                          for v in pred.values()), "prediction.json format check failed"
    with open(path, "w") as f:
        json.dump(pred, f, indent=1)
    return pred

In [ ]:
%%writefile /kaggle/working/phase7/backbones.py
"""Phase-1 checkpoints -> token feature extractors for PCR (prior case retrieval) (plumbing; declare it).

Your Phase-1 model code is used unchanged (phase1_models/: arch1/arch3 for A/B, enc_dec + parts for C,
prefix_lm + prefix_attention for D). The LM head is replaced by nn.Identity, so YOUR forward() returns the
final-LayerNorm hidden states (B, T, d_model) instead of vocabulary logits. Attention runs through the same
fastattn patches as Phase 1 (fused SDPA, identical maths; smoke_test.py re-checks equality).

Backbone options (cfg["backbone"]):
  prefix         D only: "full" -> prefix = window length (every token sees the whole window),
                 int p -> bidirectional over the first p tokens, causal after (your Phase-1 D used 640),
                 0 -> fully causal.
  encdec_mode    C only: "encoder" -> encoder states only (decoder weights dropped; bidirectional),
                 "decoder" -> the window is fed to the encoder AND to the decoder (causal self-attention +
                 cross-attention to the encoder states); features = decoder final states.
  truncate_layers keep only the first k transformer blocks (e.g. a smaller KD student); None = all.
  dropout        override every nn.Dropout p in the backbone; None = keep the checkpoint's value.
"""
import importlib
import os
import sys

import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
for _p in (HERE, os.path.join(HERE, "phase1_models")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import fastattn_ab   # noqa: E402  (A/B attention: arch1 / arch3)
import fastattn_cd   # noqa: E402  (C/D attention: encoder / decoder / cross / prefix)


def _clean_state(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("_orig_mod.", "module."):
            if k.startswith(pre):
                k = k[len(pre):]
        out[k] = v
    return out


def build_lm(spec):
    """spec = the 'config' dict saved inside a Phase-1 best.pt -> (lm, kind)."""
    kind = spec.get("kind", "decoder")
    arch = importlib.import_module(spec["module"])
    if kind == "decoder":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "prefix":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "encdec":
        lm = arch.EncDecLM(arch.EncLMConfig(**spec["encoder"]), arch.DecLMConfig(**spec["decoder"]))
    else:
        raise ValueError(f"unknown model kind {kind}")
    return lm, kind


def load_phase1(path, load_weights=True):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    spec = ck["config"]
    lm, kind = build_lm(spec)
    if load_weights:
        lm.load_state_dict(_clean_state(ck["model"]))
    info = dict(path=path, kind=kind, module=spec["module"], tag=spec.get("tag"), step=ck.get("step"),
                best_val=ck.get("best_val"))
    return lm, kind, spec, info


class Backbone(nn.Module):
    """forward(ids (B, T) long) -> hidden states (B, T, d_model).
    Attributes for technique code: .lm (your Phase-1 module), .kind, .blocks (list of transformer blocks,
    input -> output order; C "decoder" mode = encoder blocks then decoder blocks), .d_model, .max_len."""

    def __init__(self, lm, kind, prefix=None, encdec_mode=None, truncate_layers=None, dropout=None):
        super().__init__()
        self.kind = kind
        if kind == "encdec":
            assert encdec_mode in ("encoder", "decoder"), "C needs backbone.encdec_mode = 'encoder' | 'decoder'"
            self.encdec_mode = encdec_mode
            if encdec_mode == "encoder":
                for name in ("decblocks", "tok_emb2", "ln_f_dec", "head"):
                    delattr(lm, name)
                lm.ln_f_dec = lm.head = None
            else:
                lm.head = nn.Identity()
            self.d_model = lm.cfg1.d_model
            self.max_len = lm.cfg1.max_seq_len if encdec_mode == "encoder" else min(lm.cfg1.max_seq_len,
                                                                                    lm.cfg2.max_seq_len)
        else:
            if kind == "prefix":
                assert prefix is not None, "D needs backbone.prefix = 'full' | int (0 = causal)"
            self.prefix = prefix
            lm.head = nn.Identity()
            self.d_model = lm.cfg.d_model
            self.max_len = lm.cfg.max_seq_len
        self.lm = lm
        self.n_virtual_tokens = 0          # prompt tuning (Phase 3): set by your technique; see forward()
        if truncate_layers:
            self._truncate(int(truncate_layers))
        if dropout is not None:
            for m in self.modules():
                if isinstance(m, nn.Dropout):
                    m.p = float(dropout)

    @property
    def blocks(self):
        lm = self.lm
        if self.kind == "encdec":
            return list(lm.encblocks) + (list(lm.decblocks) if self.encdec_mode == "decoder" else [])
        return list(lm.blocks)

    def _truncate(self, k):
        lm = self.lm
        if self.kind == "encdec":
            assert self.encdec_mode == "encoder", "truncate_layers for C is only supported in encoder mode"
            lm.encblocks = nn.ModuleList(list(lm.encblocks)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])

    # ---- prompt-tuning hook (Phase 3 plumbing) ------------------------------------------------------------------
    # If a technique sets self.n_virtual_tokens = P > 0, forward() prepends P placeholder ids (<pad>) to every window
    # and drops the first P output states again, so callers always get (B, T, d) for their real tokens. The technique
    # makes those P positions learnable by wrapping the input embedding (get_input_embedding / set_input_embedding):
    # its wrapper returns the normal embeddings but replaces positions [:, :P] with its learned prompt vectors.
    # Positional encodings / masks see a sequence of length P + T, so windows must satisfy P + T <= max_len.
    def get_input_embedding(self):
        return self.lm.tok_emb1 if self.kind == "encdec" else self.lm.tok_emb

    def set_input_embedding(self, module):
        if self.kind == "encdec":
            self.lm.tok_emb1 = module      # encoder input (the decoder in "decoder" mode keeps its own embedding)
        else:
            self.lm.tok_emb = module

    def forward(self, ids):
        P = self.n_virtual_tokens
        if P:
            ids = torch.cat([torch.zeros(ids.shape[0], P, dtype=ids.dtype, device=ids.device), ids], dim=1)
        out = self._forward(ids)
        return out[:, P:] if P else out

    def _forward(self, ids):
        lm = self.lm
        if self.kind == "decoder":
            return lm(ids)
        if self.kind == "prefix":
            lm.cfg.prefix = ids.shape[1] if self.prefix == "full" else int(self.prefix) + self.n_virtual_tokens
            return lm(ids)
        memory = lm.encode(ids)
        if self.encdec_mode == "encoder":
            return memory
        return lm.decode(ids, memory)


def patch_fast_attention(lm, kind):
    if kind == "decoder":
        fastattn_ab.patch_model(lm)
    else:
        fastattn_cd.patch_model(lm)
    return lm


def build_backbone(ckpt_path, bcfg, vocab_size=None, load_weights=True, fast_attention=True):
    lm, kind, spec, info = load_phase1(ckpt_path, load_weights)
    if vocab_size is not None:
        v = spec["model"]["vocab_size"] if "model" in spec else spec["encoder"]["vocab_size"]
        assert v == vocab_size, f"{ckpt_path}: vocab {v} != tokenizer vocab {vocab_size} (different tokenizer?)"
    if fast_attention:
        patch_fast_attention(lm, kind)
    bb = Backbone(lm, kind, prefix=bcfg.get("prefix"), encdec_mode=bcfg.get("encdec_mode"),
                  truncate_layers=bcfg.get("truncate_layers"), dropout=bcfg.get("dropout"))
    return bb, spec, info

In [ ]:
%%writefile /kaggle/working/phase7/inject.py
"""Generic module-surgery helpers for your fine-tuning techniques (plumbing; declare it).

These only find / replace / freeze modules; they contain no adaptation method. Typical use in your files:

    from inject import freeze, replace_linears, wrap_modules
    freeze(backbone)                                            # base weights frozen
    replace_linears(backbone, lambda name, lin: name.endswith(("q_proj", "v_proj")),
                    lambda name, lin: YourWrapper(lin, ...))    # your LoRA / LoRMA module around nn.Linear
    wrap_modules(backbone, lambda name, m: name.endswith(".ffn"),
                 lambda name, m: YourAdapterWrapper(m, ...))    # your adapter after a sublayer

Linear layer names in the Phase-1 models (relative to backbone, e.g. "lm.blocks.3.attn.q_proj"):
  A/B/D blocks:  attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj, ffn.net.0 (d->d_ff), ffn.net.2 (d_ff->d)
  C encoder:     lm.encblocks.<i>.attn.{q,k,v,out}_proj, lm.encblocks.<i>.ffn.net.{0,2}
  C decoder:     lm.decblocks.<i>.attn.*, lm.decblocks.<i>.crattn.{q,k,v,out}_proj, lm.decblocks.<i>.ffn.net.{0,2}
Attention projections have no bias (bias=False); FFN linears and LayerNorms do.
Sublayer call signatures: attn(x) (D: attn(x, p)), crattn(x, memory), ffn(x) -> a wrapper should take
(*args, **kwargs) and pass them through to the wrapped module.
"""
import torch.nn as nn


def freeze(module):
    for p in module.parameters():
        p.requires_grad_(False)
    return module


def unfreeze(module):
    for p in module.parameters():
        p.requires_grad_(True)
    return module


def _set_child(root, name, new):
    parent_name, _, child = name.rpartition(".")
    parent = root.get_submodule(parent_name) if parent_name else root
    setattr(parent, child, new)


def list_linears(root):
    return [name for name, m in root.named_modules() if isinstance(m, nn.Linear)]


def replace_linears(root, match, factory):
    """Replace every nn.Linear for which match(name, linear) is True by factory(name, linear). -> replaced names."""
    targets = [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear) and match(n, m)]
    for name, lin in targets:
        _set_child(root, name, factory(name, lin))
    return [n for n, _ in targets]


def wrap_modules(root, match, factory):
    """Replace every submodule for which match(name, module) is True by factory(name, module). -> replaced names.
    Matches are collected first, so a wrapper containing the original module is not matched again."""
    targets = [(n, m) for n, m in root.named_modules() if n and match(n, m)]
    for name, mod in targets:
        _set_child(root, name, factory(name, mod))
    return [n for n, _ in targets]


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100 * trainable / max(1, total), 4)}

In [ ]:
%%writefile /kaggle/working/phase7/techniques/__init__.py
"""Technique registry (plumbing). The config's technique.name selects techniques/<name>.py.

Contract for every technique file (YOUR code, course rule 9):
    apply(backbone, **cfg) -> None
        Called once, after the Phase-1 weights are loaded and before the optimizer is built. On entry EVERY backbone
        parameter has requires_grad=True. On exit, requires_grad must be True exactly on what should train
        (plus any modules you inserted). The head is separate and always trained.
        backbone: backbones.Backbone (.lm = your Phase-1 module, .blocks, .kind, .d_model). See inject.py for
        freeze / replace_linears / wrap_modules helpers and the linear-layer names.
    optional: param_groups(model, base_lr) -> list of optimizer param-group dicts (e.g. a different lr for your
        inserted modules). If absent, every trainable backbone parameter uses train.lr and the head uses
        train.head_lr (or train.lr).
"""
import importlib


def get(name):
    return importlib.import_module(f"techniques.{name}")

In [ ]:
%%writefile /kaggle/working/phase7/techniques/full_ft.py
"""Full fine-tuning: every backbone parameter trains (plumbing-level: nothing to implement)."""


def apply(backbone, **cfg):
    assert not cfg, f"full_ft takes no options, got {cfg}"
    for p in backbone.parameters():
        p.requires_grad_(True)

In [ ]:
%%writefile /kaggle/working/phase7/train_pcr.py
#!/usr/bin/env python3
"""Fine-tune one Phase-1 model as a PCR (prior case retrieval) bi-encoder and write its Codabench submission
(plumbing; declare it). Document encoder adapted from the Phase-4/5 trainer (same chunking / pooling / techniques).

  python train_pcr.py run_config.json

Model: case text -> your ByteBPE -> up to k chunks (pcr_data.chunk_spans: even | headtail | overlap) -> Phase-1 backbone
+ YOUR technique (techniques/<name>.py) -> one vector per chunk (chunk_pool: mean | last) -> one vector per case
(doc_pool: attn = learned score per chunk, softmax over the case's chunks | mean) -> optional linear projection
(proj_dim: None = keep d_model). The SAME encoder embeds queries and candidates. score(q, c) = sim(q, c) / temperature,
sim = cosine | dot (config "similarity").

Training (one example = one (query, relevant candidate) pair of train.json, every pair once per epoch, shuffled):
  the pair's scores over its candidate set -> cross-entropy with the relevant candidate as the target (InfoNCE).
  Candidate set of a pair = its relevant candidate + negatives.per_positive negatives drawn from the query's own pool:
    source "bm25"   uniformly from the top negatives.bm25_top BM25 candidates that are not relevant (hard negatives)
    source "random" uniformly from the non-relevant pool cases
  + (negatives.in_batch = True) every other candidate document of the step. The query's OTHER relevant candidates and
  the query itself are always masked out of its row (never used as negatives).
  Gradient caching: a step embeds all its documents without gradients, takes d(loss)/d(embeddings), then re-encodes each
  micro-batch (train.micro_docs documents, same RNG state -> same dropout) and back-propagates that gradient, so the
  loss sees the whole step's documents while memory holds one micro-batch.

KD (cfg["kd"] = {"teacher_run": <finished run dir>, ...your kd options}): the teacher's embeddings of every train query
and train-pool case (its own chunking) are computed once and cached (<teacher_run>/train_emb.npz); the teacher's
scores over exactly the student's candidate rows (same masks, the teacher's similarity / temperature) and YOUR
techniques/kd.py kd_loss(student (P, 1, M), teacher (P, 1, M), target (P, 1), **opts) REPLACE the InfoNCE loss.
The student must have fewer parameters than the teacher (checked).

Evaluation: every train.eval_every_steps steps and after every epoch, dev queries are ranked against dev_candidates.json
-> Micro-F1@7 (the competition metric; selection score of best_model.pt), + P/R/F1 at other cut-offs and recall@100.
Test: test queries vs test_candidates.json -> prediction.json {query id: [7 candidate ids]} + zip.

Outputs in <out_dir>/<name>/: config.json, log.jsonl, best_model.pt, dev_scores.json, dev_S.npz / test_S.npz (full score
matrices, float32, + ids; for pcr_select.py fusion), prediction.json, submission_<name>.zip, summary.json.
"""
import inspect
import json
import math
import os
import random
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import backbones                       # noqa: E402
import pcr_data                        # noqa: E402
import techniques                      # noqa: E402
from a1_tokenizer import ByteBPE       # noqa: E402
from inject import count_params        # noqa: E402

REQUIRED = {
    "chunking": ["strategy", "max_chunks"],
    "negatives": ["source", "per_positive", "in_batch"],
    "train": ["epochs", "batch_pairs", "micro_docs", "lr", "head_lr", "weight_decay", "warmup_frac", "schedule",
              "grad_clip", "seed"],
}
MASK = -1e4


def check_config(cfg):
    missing = [f"{s}.{k}" for s, ks in REQUIRED.items() for k in ks if (cfg.get(s) or {}).get(k) is None]
    missing += [k for k in ("chunk_pool", "doc_pool", "similarity", "temperature") if cfg.get(k) is None]
    missing += ["technique.name"] if (cfg.get("technique") or {}).get("name") is None else []
    missing += ["negatives.bm25_top"] if (cfg.get("negatives") or {}).get("source") == "bm25" and \
        cfg["negatives"].get("bm25_top") is None else []
    assert not missing, "fill these config values first: " + ", ".join(missing)
    ch, tr, ng = cfg["chunking"], cfg["train"], cfg["negatives"]
    assert ch["strategy"] in ("even", "headtail", "overlap")
    assert ch["strategy"] != "overlap" or ch.get("stride"), "chunking.stride is needed for overlap"
    assert cfg["chunk_pool"] in ("mean", "last") and cfg["doc_pool"] in ("attn", "mean")
    assert cfg["similarity"] in ("cosine", "dot") and cfg["temperature"] > 0
    assert cfg.get("proj_dim") is None or int(cfg["proj_dim"]) > 0
    assert ng["source"] in ("bm25", "random") and ng["per_positive"] >= 0
    assert ng["per_positive"] > 0 or ng["in_batch"], "no negatives at all: per_positive > 0 or in_batch = True"
    assert tr["schedule"] in ("linear", "cosine", "constant")
    tech = cfg["technique"]
    unset = [f"technique.{k}" for k, v in tech.items() if v is None]
    if cfg.get("kd"):
        unset += [f"kd.{k}" for k, v in cfg["kd"].items() if v is None]
    assert not unset, "fill these config values first: " + ", ".join(unset)
    sig = inspect.signature(techniques.get(tech["name"]).apply)
    try:
        sig.bind(None, **{k: v for k, v in tech.items() if k != "name"})
    except TypeError as e:
        raise AssertionError(f"technique '{tech['name']}' options do not fit apply{sig}: {e}")


class Logger:
    def __init__(self, path):
        self.path = path

    def __call__(self, msg, **kv):
        rec = {"t": round(time.time(), 1), "msg": msg, **kv}
        print(json.dumps(rec), flush=True)
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")


# ---------------------------------------------------------------------------
# model: backbone -> chunk vectors -> case vector (-> projection)
# ---------------------------------------------------------------------------

class DocEncoder(nn.Module):
    def __init__(self, backbone, chunk_pool, doc_pool, proj_dim=None):
        super().__init__()
        self.backbone = backbone
        self.chunk_pool, self.doc_pool = chunk_pool, doc_pool
        self.attn_score = nn.Linear(backbone.d_model, 1) if doc_pool == "attn" else None
        self.proj = nn.Linear(backbone.d_model, int(proj_dim)) if proj_dim else None

    def head_modules(self):
        return [m for m in (self.attn_score, self.proj) if m is not None]

    def chunk_vectors(self, chunks, device):
        """chunks: list of 1-D int arrays -> (n_chunks, d), same order. Equal-length chunks are batched together."""
        out = [None] * len(chunks)
        by_len = {}
        for i, c in enumerate(chunks):
            by_len.setdefault(len(c), []).append(i)
        for n, idx in by_len.items():
            ids = torch.from_numpy(np.stack([chunks[i] for i in idx]).astype(np.int64)).to(device)
            h = self.backbone(ids)                                      # (b, n, d), no padding anywhere
            v = h.mean(1) if self.chunk_pool == "mean" else h[:, -1]
            for j, i in enumerate(idx):
                out[i] = v[j]
        return torch.stack(out)

    def forward(self, items, device):
        """items: list of {"chunks": [...]} -> embeddings (n_docs, d)."""
        chunks = [c for it in items for c in it["chunks"]]
        vec = self.chunk_vectors(chunks, device)
        counts = [len(it["chunks"]) for it in items]
        K, d = max(counts), vec.shape[-1]
        padded = vec.new_zeros(len(items), K, d)
        mask = torch.zeros(len(items), K, dtype=torch.bool, device=vec.device)
        k = 0
        for i, n in enumerate(counts):
            padded[i, :n] = vec[k:k + n]
            mask[i, :n] = True
            k += n
        if self.doc_pool == "attn":
            s = self.attn_score(padded).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            w = torch.softmax(s, dim=-1).to(padded.dtype)
            doc = (w.unsqueeze(-1) * padded).sum(1)
        else:
            doc = (padded * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        return self.proj(doc) if self.proj is not None else doc


def similarity(q, c, kind, temperature):
    """q (P, d), c (M, d) float32 -> scores (P, M)."""
    if kind == "cosine":
        q, c = F.normalize(q, dim=-1), F.normalize(c, dim=-1)
    return q @ c.T / temperature


def build_model(cfg, vocab_size, load_weights=True):
    bb, spec, info = backbones.build_backbone(cfg["phase1_ckpt"], cfg.get("backbone", {}), vocab_size,
                                              load_weights=load_weights, fast_attention=cfg.get("fast_attention", True))
    for p in bb.parameters():
        p.requires_grad_(True)
    tech = cfg["technique"]
    tmod = techniques.get(tech["name"])
    tmod.apply(bb, **{k: v for k, v in tech.items() if k != "name"})
    model = DocEncoder(bb, cfg["chunk_pool"], cfg["doc_pool"], cfg.get("proj_dim"))
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad_(True)
    return model, tmod, info


def chunk_len(cfg, backbone):
    limit = backbone.max_len - backbone.n_virtual_tokens          # prompt tuning prepends virtual tokens
    L = cfg["chunking"].get("chunk_len") or limit
    assert L <= limit, f"chunk_len {L} > model input limit {limit}"
    return L


def save_model(model, path):
    torch.save(model.state_dict(), path + ".tmp")
    os.replace(path + ".tmp", path)


def resolve_ckpt(cfg):
    if not os.path.exists(cfg["phase1_ckpt"]):               # earlier session: same checkpoint, new input path
        import glob
        tail = os.path.join(*cfg["phase1_ckpt"].split(os.sep)[-2:])
        hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", tail), recursive=True))
        assert hits, f"Phase-1 checkpoint {cfg['phase1_ckpt']} not found (looked for */{tail})"
        cfg["phase1_ckpt"] = hits[0]
    return cfg


def load_run(run_dir, device, vocab_size):
    cfg = resolve_ckpt(json.load(open(os.path.join(run_dir, "config.json"))))
    model, _, _ = build_model(cfg, vocab_size, load_weights=False)
    model.load_state_dict(torch.load(os.path.join(run_dir, "best_model.pt"), map_location="cpu"))
    return model.to(device).eval(), cfg


# ---------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------

def load_data(cfg, tok):
    D, cache = cfg["data_dir"], cfg.get("cache_dir")
    data, arrays, rep = {}, {}, {}
    for s in ("train", "dev", "test"):
        q, pool = pcr_data.load_split(D, s)
        data[s] = {"queries": q, "pool": pool}
        rep[s] = pcr_data.split_stats(q, pool)
    qh = {s: {pcr_data.norm_hash(d["text"]) for d in data[s]["queries"]} for s in data}
    rep["train_queries_equal_dev_queries"] = len(qh["train"] & qh["dev"])
    rep["train_queries_equal_test_queries"] = len(qh["train"] & qh["test"])
    for s in data:
        for part in ("queries", "pool"):
            docs = data[s][part]
            arrays[(s, part)] = pcr_data.encode_docs(tok, docs, os.path.join(cache, f"{s}_{part}_ids.npz") if cache else None)
    return data, arrays, rep


def smoke_subset(data, arrays, n_train, seed=0):
    """few real queries + small pools (their relevant cases + random others); not submittable."""
    rnd = random.Random(seed)
    for s, nq, extra in (("train", n_train, 48), ("dev", 4, 24), ("test", 4, 24)):
        qs, pool = data[s]["queries"], data[s]["pool"]
        qi = sorted(range(len(qs)), key=lambda i: -len(arrays[(s, "queries")][i]))[:nq]
        keep = {c for i in qi for c in (qs[i]["relevant"] or [])}
        others = [j for j, d in enumerate(pool) if d["id"] not in keep]
        pj = [j for j, d in enumerate(pool) if d["id"] in keep] + rnd.sample(others, min(extra, len(others)))
        data[s] = {"queries": [qs[i] for i in qi], "pool": [pool[j] for j in pj]}
        arrays[(s, "queries")] = [arrays[(s, "queries")][i] for i in qi]
        arrays[(s, "pool")] = [arrays[(s, "pool")][j] for j in pj]


def make_items(docs, arr, cfg, L):
    ch = cfg["chunking"]
    return pcr_data.build_items(docs, arr, L, ch["strategy"], ch["max_chunks"], ch.get("stride"))


# ---------------------------------------------------------------------------
# inference
# ---------------------------------------------------------------------------

@torch.no_grad()
def embed(model, items, micro, device, amp):
    """-> float32 tensor (n, d) on device, same order as items (micro-batches grouped by chunk count)."""
    model.eval()
    out = [None] * len(items)
    order = sorted(range(len(items)), key=lambda i: len(items[i]["chunks"]))
    for a in range(0, len(order), micro):
        idx = order[a:a + micro]
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            e = model([items[i] for i in idx], device).float()
        for j, i in enumerate(idx):
            out[i] = e[j]
    return torch.stack(out)


def score_matrix(model, q_items, p_items, cfg, micro, device, amp):
    qe, pe = embed(model, q_items, micro, device, amp), embed(model, p_items, micro, device, amp)
    return similarity(qe, pe, cfg["similarity"], cfg["temperature"]).cpu().numpy()


def grad_cache_backward(model, docs, micro, device, amp, loss_fn, scale=lambda x: x):
    """Gradient caching: loss over the embeddings of ALL docs, memory of one micro-batch.
    pass 1 embeds every micro-batch without gradients (RNG state saved), loss_fn(E (n, d) float32) is back-propagated to
    E only, pass 2 re-encodes each micro-batch with gradients under the same RNG state (identical dropout masks) and
    back-propagates dL/dE into the model. -> (loss, tokens encoded with gradients)."""
    def rng_state():
        return (torch.get_rng_state(), torch.cuda.get_rng_state(device) if device.type == "cuda" else None)

    def set_rng(st):
        torch.set_rng_state(st[0])
        if st[1] is not None:
            torch.cuda.set_rng_state(st[1], device)

    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["chunks"]))
    mbs = [order[i:i + micro] for i in range(0, len(order), micro)]
    states, E = [], [None] * len(docs)
    with torch.no_grad():
        for mb in mbs:
            states.append(rng_state())
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                e = model([docs[i] for i in mb], device).float()
            for j, i in enumerate(mb):
                E[i] = e[j]
    E = torch.stack(E).detach().requires_grad_(True)
    loss = loss_fn(E)
    scale(loss).backward()
    G, ntok = E.grad, 0
    end_state = rng_state()
    for mb, st in zip(mbs, states):
        set_rng(st)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            e = model([docs[i] for i in mb], device).float()
        e.backward(G[mb])
        ntok += sum(len(c) for i in mb for c in docs[i]["chunks"])
    set_rng(end_state)
    return loss.detach(), ntok


def gold_of(queries):
    return {q["id"]: q["relevant"] for q in queries}


def lr_factor(step, total, warmup, schedule):
    if warmup and step < warmup:
        return (step + 1) / warmup
    if schedule == "constant":
        return 1.0
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return 1.0 - prog if schedule == "linear" else 0.5 * (1 + math.cos(math.pi * prog))


def bm25_top(data, split, top, cache_dir, log=None):
    """{query id: [pool ids by BM25 score, best first, the query itself excluded]} (top `top`), cached."""
    path = os.path.join(cache_dir, f"bm25_{split}_top{top}.json") if cache_dir else None
    qids = [q["id"] for q in data[split]["queries"]]
    if path and os.path.exists(path):
        c = json.load(open(path))
        if sorted(c) == sorted(qids):
            return c
    t0 = time.time()
    bm = pcr_data.BM25(data[split]["pool"])
    S = bm.score_matrix(data[split]["queries"])
    out = pcr_data.rank(S, qids, bm.ids, min(top, len(bm.ids)))
    if path:
        json.dump(out, open(path, "w"))
    if log:
        log("bm25 ranking", split=split, top=top, seconds=round(time.time() - t0, 1))
    return out


def teacher_embeddings(cfg, data, arrays, device, amp, vocab_size, log):
    """teacher embeddings of every train query and train-pool case (teacher's own chunking), cached in the teacher dir."""
    tdir = cfg["kd"]["teacher_run"]
    qids = [d["id"] for d in data["train"]["queries"]]
    pids = [d["id"] for d in data["train"]["pool"]]
    for path in (os.path.join(tdir, "train_emb.npz"), os.path.join(cfg["out_dir"], cfg["name"], "teacher_train_emb.npz")):
        if os.path.exists(path):
            z = np.load(path)
            if list(z["qids"]) == qids and list(z["pids"]) == pids:
                log("teacher embeddings: cached", path=path)
                return {"q": z["q"], "p": z["p"]}
    teacher, tcfg = load_run(tdir, device, vocab_size)
    L = chunk_len(tcfg, teacher.backbone)
    im = tcfg["train"].get("infer_micro_docs") or 4
    t0 = time.time()
    qe = embed(teacher, make_items(data["train"]["queries"], arrays[("train", "queries")], tcfg, L), im, device, amp)
    pe = embed(teacher, make_items(data["train"]["pool"], arrays[("train", "pool")], tcfg, L), im, device, amp)
    emb = {"q": qe.cpu().numpy(), "p": pe.cpu().numpy()}
    path = os.path.join(tdir, "train_emb.npz")
    try:
        np.savez(path + ".tmp.npz", qids=np.array(qids), pids=np.array(pids), **emb)
        os.replace(path + ".tmp.npz", path)
    except OSError:                                   # teacher folder read-only (an attached Kaggle input)
        path = os.path.join(cfg["out_dir"], cfg["name"], "teacher_train_emb.npz")
        np.savez(path, qids=np.array(qids), pids=np.array(pids), **emb)
    log("teacher embeddings: computed", path=path, seconds=round(time.time() - t0, 1), teacher=tcfg["name"])
    del teacher
    torch.cuda.empty_cache()
    return emb


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def main(cfg_path):
    cfg = json.load(open(cfg_path))
    check_config(cfg)
    name, tr, ng = cfg["name"], cfg["train"], cfg["negatives"]
    out = os.path.join(cfg["out_dir"], name)
    os.makedirs(out, exist_ok=True)
    log = Logger(os.path.join(out, "log.jsonl"))
    json.dump(cfg, open(os.path.join(out, "config.json"), "w"), indent=2)
    seed = tr.get("seed", 0)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    t_start = time.time()
    deadline = t_start + 60 * tr["max_minutes"] if tr.get("max_minutes") else None

    tok = ByteBPE.from_json(cfg["tokenizer"])
    data, arrays, drep = load_data(cfg, tok)
    smoke = cfg.get("smoke")            # {"steps": n, "queries": m}: real model, few queries, small pools
    if smoke:
        smoke_subset(data, arrays, smoke.get("queries", 8), seed)
        tr["epochs"] = 1

    model, tmod, info = build_model(cfg, tok.get_vocab_size())
    model.to(device)
    L = chunk_len(cfg, model.backbone)
    items = {k: make_items(data[k[0]][k[1]], arrays[k], cfg, L) for k in arrays}
    log("data", **{f"{s}_{k}": v for s in ("train", "dev", "test") for k, v in drep[s].items()},
        train_queries_equal_dev_queries=drep["train_queries_equal_dev_queries"],
        train_queries_equal_test_queries=drep["train_queries_equal_test_queries"], chunk_len=L,
        strategy=cfg["chunking"]["strategy"], max_chunks=cfg["chunking"]["max_chunks"],
        **{f"{s}_{p}_coverage": round(pcr_data.coverage(v), 4) for (s, p), v in items.items()})
    params = count_params(model)
    params["head"] = sum(p.numel() for m in model.head_modules() for p in m.parameters())
    log("model", **info, **params, chunk_len=L, max_len=model.backbone.max_len)

    tq, tp = data["train"]["queries"], data["train"]["pool"]
    pidx = {d["id"]: j for j, d in enumerate(tp)}
    pairs = [(i, pidx[c]) for i, q in enumerate(tq) for c in q["relevant"] if c in pidx]
    rel_idx = [{pidx[c] for c in q["relevant"] if c in pidx} for q in tq]
    self_idx = [pidx.get(q["id"]) for q in tq]
    assert pairs, "no (query, relevant candidate) pair found in the train pool"
    hard = None
    if ng["source"] == "bm25" and ng["per_positive"]:
        top = bm25_top(data, "train", int(ng["bm25_top"]) + 64, None if smoke else cfg.get("cache_dir"), log)
        hard = []
        for i, q in enumerate(tq):
            cand = [pidx[c] for c in top[q["id"]] if pidx[c] not in rel_idx[i] and pidx[c] != self_idx[i]]
            hard.append(cand[:int(ng["bm25_top"])])

    t_emb = None
    if cfg.get("kd"):
        kd = {k: v for k, v in cfg["kd"].items() if k != "teacher_run"}
        tcfg = json.load(open(os.path.join(cfg["kd"]["teacher_run"], "config.json")))
        tsum = json.load(open(os.path.join(cfg["kd"]["teacher_run"], "summary.json")))
        assert params["total"] < tsum["params"]["total"], \
            f"KD student ({params['total']:,}) must be smaller than the teacher ({tsum['params']['total']:,})"
        t_emb = teacher_embeddings(cfg, data, arrays, device, amp, tok.get_vocab_size(), log)
        t_emb = {k: torch.from_numpy(v).to(device) for k, v in t_emb.items()}
        kd_loss = techniques.get("kd").kd_loss
        log("teacher", run=tcfg["name"], params=tsum["params"]["total"], student_params=params["total"],
            teacher_dev_f1_at7=tsum.get("dev_f1_at7"))

    if hasattr(tmod, "param_groups"):
        groups = tmod.param_groups(model, tr["lr"])
    else:
        head_params = [p for m in model.head_modules() for p in m.parameters()]
        groups = [{"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": tr["lr"]},
                  {"params": head_params, "lr": tr["head_lr"]}]
    for g in groups:
        g["params"] = list(g["params"])
        g["base_lr"] = g["lr"]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=tr["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    extra_loss, on_step = getattr(tmod, "extra_loss", None), getattr(tmod, "on_step", None)

    rng = random.Random(seed)
    B, micro = tr["batch_pairs"], tr["micro_docs"]
    steps_per_epoch = math.ceil(len(pairs) / B)
    total = steps_per_epoch * tr["epochs"]
    warmup = int(tr["warmup_frac"] * total)
    log("schedule", pairs=len(pairs), steps_per_epoch=steps_per_epoch, total_steps=total, warmup_steps=warmup,
        pairs_per_step=B, negatives=ng)

    def sample_negatives(i):
        n = ng["per_positive"]
        if not n:
            return []
        if hard is not None and hard[i]:
            pool_ = hard[i]
        else:
            pool_ = [j for j in range(len(tp)) if j not in rel_idx[i] and j != self_idx[i]]
        return rng.sample(pool_, min(n, len(pool_)))

    def step_rows(batch):
        """-> query rows, candidate columns, target per row, allowed mask (P, M)."""
        cand, col = [], {}
        rows = []
        for qi, pj in batch:
            negs = sample_negatives(qi)
            for j in [pj] + negs:
                if j not in col:
                    col[j] = len(cand); cand.append(j)
            rows.append((qi, pj, negs))
        P, M = len(rows), len(cand)
        allowed = torch.zeros(P, M, dtype=torch.bool)
        target = torch.zeros(P, dtype=torch.long)
        for r, (qi, pj, negs) in enumerate(rows):
            if ng["in_batch"]:
                allowed[r] = True
            else:
                allowed[r, [col[j] for j in [pj] + negs]] = True
            for j in rel_idx[qi] | ({self_idx[qi]} if self_idx[qi] is not None else set()):
                if j != pj and j in col:
                    allowed[r, col[j]] = False
            allowed[r, col[pj]] = True
            target[r] = col[pj]
        return [qi for qi, _, _ in rows], cand, target, allowed

    sel = {"best": -1.0, "at": None, "epoch": -1}
    history = []
    im = tr.get("infer_micro_docs") or 4
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    def evaluate(tag, epoch, rec):
        t_ev = time.time()
        S = score_matrix(model, items[("dev", "queries")], items[("dev", "pool")], cfg, im, device, amp)
        rep = pcr_data.report(S, [q["id"] for q in data["dev"]["queries"]], [d["id"] for d in data["dev"]["pool"]],
                              gold_of(data["dev"]["queries"]))
        rec.update(at=tag, dev_f1_at7=rep["at7"]["micro_f1"], dev_p_at7=rep["at7"]["micro_p"],
                   dev_r_at7=rep["at7"]["micro_r"], dev_r_at100=rep["recall@100"], eval_s=round(time.time() - t_ev, 1))
        if rep["at7"]["micro_f1"] > sel["best"]:
            sel.update(best=rep["at7"]["micro_f1"], at=tag, epoch=epoch)
            save_model(model, os.path.join(out, "best_model.pt"))
            rec["saved_best"] = True
        history.append(rec)
        model.train()
        return rec

    step, seen_tok, epochs_done, stop = 0, 0, 0, False
    t_train = time.time()
    eval_every = tr.get("eval_every_steps")
    if not smoke and tr.get("eval_at_start", True):
        log("eval", **evaluate("start", 0, dict(epoch=0, step=0)))        # Phase-1 representations, no PCR training
    for epoch in range(1, tr["epochs"] + 1):
        model.train()
        order = pairs[:]
        rng.shuffle(order)
        run_loss, n_loss, t_ep, complete = 0.0, 0, time.time(), True
        for a in range(0, len(order), B):
            batch = order[a:a + B]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(step, total, warmup, tr["schedule"])
            q_rows, cand, target, allowed = step_rows(batch)
            uq = sorted(set(q_rows))
            docs = [items[("train", "queries")][i] for i in uq] + [items[("train", "pool")][j] for j in cand]
            qpos = {qi: k for k, qi in enumerate(uq)}
            am, tgt = allowed.to(device), target.to(device)

            def step_loss(E):
                S = similarity(E[[qpos[qi] for qi in q_rows]], E[len(uq):], cfg["similarity"], cfg["temperature"])
                S = S.masked_fill(~am, MASK)
                if t_emb is not None:
                    St = similarity(t_emb["q"][q_rows], t_emb["p"][cand], tcfg["similarity"], tcfg["temperature"])
                    return kd_loss(S.unsqueeze(1), St.masked_fill(~am, MASK).unsqueeze(1), tgt.unsqueeze(1), **kd)
                return F.cross_entropy(S, tgt)

            loss, ntok = grad_cache_backward(model, docs, micro, device, amp, step_loss, scaler.scale)
            seen_tok += ntok
            if extra_loss is not None:
                scaler.scale(extra_loss(model)).backward()
            if tr["grad_clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], tr["grad_clip"])
            scaler.step(opt); scaler.update()
            step += 1
            if on_step is not None:
                on_step(model, step, total)
            opt.zero_grad(set_to_none=True)
            run_loss += loss.item(); n_loss += 1
            if smoke and step >= smoke["steps"]:
                stop = True
                break
            if eval_every and step % eval_every == 0 and step % steps_per_epoch != 0:
                log("eval", **evaluate(f"e{epoch}_step{step}", epoch,
                                       dict(epoch=epoch, step=step, train_loss=round(run_loss / max(1, n_loss), 4))))
            if step % tr.get("log_every", 20) == 0:
                el = time.time() - t_train
                log("train", epoch=epoch, step=step, of=total, loss=round(run_loss / max(1, n_loss), 4),
                    lr=opt.param_groups[0]["lr"], tok_s=round(seen_tok / el), docs_in_step=len(docs),
                    epoch_eta_min=round((steps_per_epoch * epoch - step) * el / step / 60, 1))
            if deadline and time.time() > deadline:
                log("max_minutes reached: stopping inside the epoch (this epoch is INCOMPLETE)", step=step)
                stop, complete = True, False
                break
        if complete and not smoke:
            epochs_done = epoch
        rec = dict(epoch=epoch, complete_epoch=complete and not smoke, step=step,
                   train_loss=round(run_loss / max(1, n_loss), 4), epoch_s=round(time.time() - t_ep, 1))
        log("epoch", **evaluate(f"e{epoch}_end", epoch, rec))
        if stop or (tr.get("patience") and epoch - sel["epoch"] >= tr["patience"]):
            break
    train_s = time.time() - t_train
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None

    model.load_state_dict(torch.load(os.path.join(out, "best_model.pt"), map_location="cpu"))
    dq, dpool = [q["id"] for q in data["dev"]["queries"]], [d["id"] for d in data["dev"]["pool"]]
    S_dev = score_matrix(model, items[("dev", "queries")], items[("dev", "pool")], cfg, im, device, amp)
    dev_rep = pcr_data.report(S_dev, dq, dpool, gold_of(data["dev"]["queries"]))
    json.dump(dev_rep, open(os.path.join(out, "dev_scores.json"), "w"), indent=2)
    np.savez(os.path.join(out, "dev_S.npz"), S=S_dev.astype(np.float32), qids=np.array(dq), cids=np.array(dpool))
    tq_ids, tpool = [q["id"] for q in data["test"]["queries"]], [d["id"] for d in data["test"]["pool"]]
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_inf = time.time()
    S_test = score_matrix(model, items[("test", "queries")], items[("test", "pool")], cfg, im, device, amp)
    inf_s = time.time() - t_inf
    np.savez(os.path.join(out, "test_S.npz"), S=S_test.astype(np.float32), qids=np.array(tq_ids), cids=np.array(tpool))
    pred_path = os.path.join(out, "prediction.json")
    pcr_data.write_prediction(S_test, tq_ids, tpool, pred_path)
    zip_path = os.path.join(out, f"submission_{name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    n_test_docs = len(tq_ids) + len(tpool)
    summary = dict(name=name, model_key=cfg.get("model_key"), phase1=info, technique=cfg["technique"],
                   kd=bool(cfg.get("kd")), similarity=cfg["similarity"], temperature=cfg["temperature"],
                   proj_dim=cfg.get("proj_dim"), chunk_pool=cfg["chunk_pool"], doc_pool=cfg["doc_pool"],
                   chunking={**cfg["chunking"], "chunk_len": L}, negatives=ng, lr=tr["lr"], head_lr=tr["head_lr"],
                   params=params, selected_at=sel["at"], best_epoch=sel["epoch"], epochs_completed=epochs_done,
                   full_epoch_completed=epochs_done >= 1, history=history,
                   dev_f1_at7=dev_rep["at7"]["micro_f1"], dev=dev_rep,
                   start_dev_f1_at7=history[0]["dev_f1_at7"] if history and history[0]["at"] == "start" else None,
                   train_seconds=round(train_s, 1), train_tok_s=round(seen_tok / max(train_s, 1e-9)),
                   peak_mem_GB=round(peak, 2) if peak else None, test_infer_seconds=round(inf_s, 1),
                   test_docs_encoded=n_test_docs, infer_ms_per_doc=round(1000 * inf_s / max(1, n_test_docs), 1),
                   total_seconds=round(time.time() - t_start, 1), data=drep, smoke=bool(smoke))
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2)
    log("done", name=name, dev_f1_at7=dev_rep["at7"]["micro_f1"], dev_p_at7=dev_rep["at7"]["micro_p"],
        dev_r_at7=dev_rep["at7"]["micro_r"], selected_at=sel["at"], epochs_completed=epochs_done,
        train_min=round(train_s / 60, 1), peak_GB=summary["peak_mem_GB"], trainable=params["trainable"], zip=zip_path)


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/phase7/run_many.py
#!/usr/bin/env python3
"""Run several train_pcr.py configs over the available GPUs IN PARALLEL (plumbing; declare it).

  python run_many.py --gpus 0,1 configs/B_full.json configs/C_full.json ...

One worker per GPU; every worker runs one training process at a time on its own GPU (CUDA_VISIBLE_DEVICES), so with
--gpus 0,1 two fine-tuning runs train simultaneously, one per T4. Scheduling uses a SHARED queue: whenever a GPU becomes
free it takes the first runnable config in list order (no GPU sits idle while work remains). A config with "kd" is only
runnable once its teacher has written summary.json; until then the free GPU takes the next job instead. If the teacher
fails (or is not part of this batch and has no summary.json), the KD run is marked FAILED instead of waiting forever.
With --stop-at UNIX_TIME a run is only STARTED if it can finish before then (its train.max_minutes + --margin minutes,
or --default-minutes when max_minutes is unset); runs that no longer fit are marked SKIPPED, so the session limit never
kills a run halfway.

Logs: <out_dir>/<name>.stdout (+ <out_dir>/<name>/log.jsonl). A status line every --every seconds, a summary at the end.
Exit code 1 if any run failed.
"""
import argparse
import json
import os
import subprocess
import sys
import threading
import time

HERE = os.path.dirname(os.path.abspath(__file__))


class Scheduler:
    def __init__(self, cfg_paths, stop_at=None, margin=10, default_minutes=60):
        self.stop_at, self.margin, self.default_minutes = stop_at, margin, default_minutes
        self.lock = threading.Lock()
        self.cfgs = {}
        for p in cfg_paths:
            c = json.load(open(p))
            self.cfgs[c["name"]] = (p, c)
        self.pending = list(self.cfgs)                        # names, in the order given
        self.status = {n: "queued" for n in self.pending}
        self.started = {}

    def _teacher(self, cfg):
        if not cfg.get("kd"):
            return None, None
        d = cfg["kd"]["teacher_run"].rstrip("/")
        return os.path.basename(d), os.path.join(d, "summary.json")

    def next_job(self):
        """-> (name, path, cfg) to run now | "wait" (only KD runs whose teacher is still training) | None (nothing left)."""
        with self.lock:
            if not self.pending:
                return None
            for name in list(self.pending):
                path, cfg = self.cfgs[name]
                teacher, done = self._teacher(cfg)
                if teacher is not None and not os.path.exists(done):
                    t_status = self.status.get(teacher)
                    if t_status is None or t_status.startswith(("FAILED", "SKIPPED", "done")):
                        self.pending.remove(name)
                        self.status[name] = f"FAILED (teacher {teacher} has no summary.json)"
                        continue
                    self.status[name] = f"waiting for teacher {teacher}"
                    continue
                if self.stop_at is not None:
                    need = ((cfg.get("train", {}).get("max_minutes") or cfg.get("est_minutes") or self.default_minutes) + self.margin) * 60
                    if time.time() + need > self.stop_at:
                        self.pending.remove(name)
                        self.status[name] = f"SKIPPED (needs ~{need / 60:.0f} min, session limit too close)"
                        continue
                self.pending.remove(name)
                return name, path, cfg
            return "wait" if self.pending else None

    def worker(self, gpu):
        while True:
            job = self.next_job()
            if job is None:
                return
            if job == "wait":
                time.sleep(10)
                continue
            name, path, cfg = job
            os.makedirs(cfg["out_dir"], exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2")
            with self.lock:
                self.status[name] = f"running on GPU {gpu}"
                self.started[name] = time.time()
            with open(os.path.join(cfg["out_dir"], f"{name}.stdout"), "w") as log:
                rc = subprocess.call([sys.executable, os.path.join(HERE, "train_pcr.py"), path], env=env,
                                     stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
            mins = (time.time() - self.started[name]) / 60
            if rc != 0:                                  # show WHY it failed right in the main log
                try:
                    tail = open(os.path.join(cfg["out_dir"], f"{name}.stdout"), errors="replace").read().strip().splitlines()[-25:]
                    print(f"\n----- {name} FAILED (exit {rc}): last lines of its output -----\n" + "\n".join(tail) +
                          "\n-----", flush=True)
                except OSError:
                    pass
            with self.lock:
                self.status[name] = f"done on GPU {gpu} in {mins:.1f} min" if rc == 0 else \
                    f"FAILED (exit {rc}) on GPU {gpu} after {mins:.1f} min"


def last_log_line(cfg):
    logf = os.path.join(cfg["out_dir"], cfg["name"], "log.jsonl")
    if not os.path.exists(logf):
        return None
    lines = open(logf).read().strip().splitlines()
    return lines[-1][:250] if lines else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("configs", nargs="+")
    ap.add_argument("--gpus", default="0")
    ap.add_argument("--every", type=int, default=300)
    ap.add_argument("--stop-at", type=float, default=None, help="unix time: do not start runs that cannot finish by then")
    ap.add_argument("--margin", type=float, default=10, help="minutes added to max_minutes for evaluation + prediction")
    ap.add_argument("--default-minutes", type=float, default=60, help="assumed run length when max_minutes is unset")
    args = ap.parse_args()
    gpus = [g for g in args.gpus.split(",") if g != ""]
    sched = Scheduler(args.configs, args.stop_at, args.margin, args.default_minutes)
    print(f"{len(sched.cfgs)} runs on GPUs {gpus} ({len(gpus)} in parallel): {list(sched.cfgs)}", flush=True)
    threads = [threading.Thread(target=sched.worker, args=(g,), daemon=True, name=f"gpu{g}") for g in gpus]
    for t in threads:
        t.start()
    t0 = time.time()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=args.every / len(threads))
        with sched.lock:
            status = dict(sched.status)
        print(f"\n{time.strftime('%H:%M:%S')} (+{(time.time() - t0) / 60:.1f} min)", flush=True)
        for name, st in status.items():
            line = last_log_line(sched.cfgs[name][1]) if st.startswith("running") else None
            print(f"  {name:<20} {st}" + (f"\n      {line}" if line else ""), flush=True)

    print("\nsummary")
    failed = []
    for name, (path, cfg) in sched.cfgs.items():
        sp = os.path.join(cfg["out_dir"], name, "summary.json")
        if os.path.exists(sp):
            s = json.load(open(sp))
            print(f"  {name:<22} dev Micro-F1@7 {s['dev_f1_at7']}  (untrained start {s.get('start_dev_f1_at7')})  "
                  f"epochs {s['epochs_completed']}  trainable {s['params']['trainable_pct']}%  train {s['train_seconds'] / 60:.1f} min  "
                  f"peak {s['peak_mem_GB']} GB  ({sched.status[name]})")
        else:
            print(f"  {name:<20} no summary ({sched.status[name]})")
        if not sched.status[name].startswith("done"):
            failed.append(name)
    if failed:
        sys.exit(f"failed runs: {failed} (see <out_dir>/<name>.stdout)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase7/pcr_select.py
#!/usr/bin/env python3
"""Score fusion of finished PCR runs (and optionally BM25) -> dev Micro-F1@7 + test submission (plumbing; declare it).

  python pcr_select.py --name ENS1 --out DIR --data-dir DATA --members RUN_DIR [RUN_DIR ...] [--weights w1 w2 ...]
                       [--bm25-weight W] [--norm zscore|minmax|none]

Each member's dev_S.npz / test_S.npz (written by train_pcr.py) is normalised per query row (--norm), multiplied by its
weight and summed; --bm25-weight W adds BM25 scores of the same split (same normalisation). Weights are YOUR values
(course rule 9); nothing is tuned here. Only one member and --bm25-weight 0 = that run unchanged.
  python pcr_select.py --bm25-only --data-dir DATA --out DIR      # local BM25 reference score on dev (not a submission)
Outputs in DIR/<name>/: dev_scores.json, prediction.json, submission_<name>.zip.
"""
import argparse
import json
import os
import sys
import zipfile

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pcr_data  # noqa: E402


def norm_rows(S, how):
    S = S.astype(np.float64)
    if how == "zscore":
        return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
    if how == "minmax":
        lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
        return (S - lo) / (hi - lo + 1e-9)
    return S


def load_S(run_dir, split, qids, cids):
    z = np.load(os.path.join(run_dir, f"{split}_S.npz"))
    q, c = list(z["qids"]), list(z["cids"])
    qi, ci = {x: i for i, x in enumerate(q)}, {x: j for j, x in enumerate(c)}
    assert set(q) == set(qids) and set(c) == set(cids), f"{run_dir}: {split} ids differ from the data files"
    return z["S"][np.ix_([qi[x] for x in qids], [ci[x] for x in cids])]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="BM25")
    ap.add_argument("--out", required=True)
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--members", nargs="*", default=[])
    ap.add_argument("--weights", nargs="*", type=float, default=None)
    ap.add_argument("--bm25-weight", type=float, default=0.0)
    ap.add_argument("--bm25-only", action="store_true")
    ap.add_argument("--norm", default="zscore", choices=["zscore", "minmax", "none"])
    a = ap.parse_args()
    assert a.members or a.bm25_only or a.bm25_weight, "give --members and/or --bm25-weight, or --bm25-only"
    w = a.weights or [1.0] * len(a.members)
    assert len(w) == len(a.members), "one weight per member"
    od = os.path.join(a.out, a.name)
    os.makedirs(od, exist_ok=True)
    res = {"members": a.members, "weights": w, "bm25_weight": 1.0 if a.bm25_only else a.bm25_weight, "norm": a.norm}
    for split in (("dev",) if a.bm25_only else ("dev", "test")):
        queries, pool = pcr_data.load_split(a.data_dir, split)
        qids, cids = [q["id"] for q in queries], [d["id"] for d in pool]
        S = np.zeros((len(qids), len(cids)))
        for m, wm in zip(a.members, w):
            S += wm * norm_rows(load_S(m, split, qids, cids), a.norm)
        if a.bm25_only or a.bm25_weight:
            S += res["bm25_weight"] * norm_rows(pcr_data.BM25(pool).score_matrix(queries), a.norm)
        if split == "dev":
            res["dev"] = pcr_data.report(S, qids, cids, {q["id"]: q["relevant"] for q in queries})
            print(json.dumps({"name": a.name, **res["dev"]["at7"], "curve": res["dev"]["curve"],
                              "recall@100": res["dev"]["recall@100"]}))
        else:
            pp = os.path.join(od, "prediction.json")
            pcr_data.write_prediction(S, qids, cids, pp)
            with zipfile.ZipFile(os.path.join(od, f"submission_{a.name}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
                z.write(pp, arcname="prediction.json")
    json.dump(res, open(os.path.join(od, "dev_scores.json"), "w"), indent=2)


if __name__ == "__main__":
    main()

## Stage 1b — YOUR files: techniques, experiment matrix

In [ ]:
%%writefile /kaggle/working/phase7/techniques/partial_ft.py
"""YOUR FILE: Partial / layer-wise fine-tuning.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); then unfreeze the parts you choose (backbone.blocks is input -> output order)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone,k):
    freeze(backbone)
    if(k>0):
        
    # unfreeze the final norm layer
        if(backbone.kind == "encdec"):
            if backbone.encdec_mode == "encoder":
                unfreeze(backbone.lm.ln_f_enc)
            else:
                unfreeze(backbone.lm.ln_f_dec)


            
           
        else:
            unfreeze(backbone.lm.ln_f)
        for block in  backbone.blocks[-k:]:
            unfreeze(block)
        
        
    
    




   

In [ ]:
%%writefile /kaggle/working/phase7/techniques/qlora.py
"""YOUR FILE: QLoRA — Dettmers et al. (2023) [9].

WHAT IT IS
  LoRA on top of a QUANTISED frozen base: the frozen W0 of every adapted linear is stored in 4 bits and dequantised
  on the fly in the forward pass; only the LoRA A/B matrices (kept in 16/32-bit) train. The paper's pieces:
    * NF4 ("4-bit NormalFloat"): 16 fixed quantisation levels placed at quantiles of a standard normal, so normally
      distributed weights use all 16 levels evenly;
    * block-wise quantisation: weights split into blocks (e.g. 64 values); each block stores its absmax as the scale,
      values stored as the index (0..15) of the nearest NF4 level after dividing by absmax;
    * double quantisation (optional): the per-block absmax constants are themselves quantised (8-bit) to save memory;
    * paged optimizers (memory paging) — not needed here.
  Forward: W0_hat = dequant(codes, absmax);  h = x @ W0_hat^T + bias + scale * B(A(x)).

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "qlora", "r": ..., "alpha": ..., "targets": [...], "block_size": ...}

SUGGESTED STRUCTURE (from scratch, PyTorch only)
  NF4_LEVELS = torch.tensor([...16 values...])           # the paper / bitsandbytes NF4 table
  def quantize_nf4(w, block_size) -> (codes uint8, absmax float)   # per block: absmax, then nearest-level index
  def dequantize_nf4(codes, absmax, shape, block_size) -> tensor
  class QLoRALinear(nn.Module):
      __init__(self, base: nn.Linear, r, alpha, block_size):
          quantise base.weight once; register codes / absmax as BUFFERS (not parameters); drop the fp weight
          create LoRA A, B as in your lora.py (B = 0)
      forward(self, x): F.linear(x, dequantize(...).to(x.dtype), base.bias) + scaling * B(A(x))
  def apply(backbone, r, alpha, targets, block_size): freeze(backbone); replace_linears(... -> QLoRALinear ...)
  Note: the quantisation error of the frozen weights is part of the method; compare memory (peak_mem_GB in
  summary.json) with plain LoRA. (Using the bitsandbytes library instead would be a library implementation — if you
  do, cite it and declare it.)
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from inject import freeze, unfreeze, replace_linears   # noqa: F401

NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367,
    -0.39491748809814453, -0.28444138169288635,
    -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725,
    0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0
], dtype=torch.float32)
def quantize_nf4(w,block_size):
  assert w.numel() % block_size == 0
  blocks = w.reshape(-1,block_size)
  absmax = blocks.abs().max(dim=1).values.clamp(min=1e-12)
  blocks = blocks/absmax[:, None]
  dist  = (blocks[..., None] - NF4_LEVELS.to(blocks.device)).abs()
  codes = dist.argmin(dim=-1).to(torch.uint8)
  return (codes,absmax)
def dequantize_nf4(codes,absmax,shape,block_size):
  output = NF4_LEVELS.to(codes.device)[codes.long()]
  output = output*absmax[:, None]
  return output.reshape(shape)
class QLoRALinear(nn.Module):
  def __init__(self,base:nn.Linear,r,alpha,block_size):
    super().__init__()
    self.r = r 
    self.alpha=alpha
    self.block_size=block_size
    codes, absmax = quantize_nf4(base.weight.detach(), block_size)
    self.register_buffer("codes", codes)
    self.register_buffer("absmax", absmax)
    self.in_features=base.in_features
    self.out_features=base.out_features
    self.shape = tuple(base.weight.shape)
    if base.bias is not None:
      self.register_buffer("bias", base.bias.detach().clone())
    else:
      self.bias = None
    self.A = nn.Parameter(torch.randn(r, self.in_features))
        
    self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    self.B = nn.Parameter(torch.zeros(self.out_features,r))
    self.scaling  = alpha/r
  def forward(self,x:torch.Tensor):
    base = F.linear(x, dequantize_nf4(self.codes,self.absmax,self.shape,self.block_size).to(x.dtype), self.bias)
    lora  = (x@self.A.T)@self.B.T
    return base+self.scaling*lora











def apply(backbone, r, alpha, targets,block_size):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: QLoRALinear(lin, r, alpha,block_size))

In [ ]:
%%writefile /kaggle/working/phase7/techniques/lorma.py
"""YOUR FILE: LoRMA: Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, 2025) [12] -- COMPULSORY.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRMALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRMALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.empty(r, self.out_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (base_output@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRMALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase7/techniques/prefix_tuning.py
"""YOUR FILE: Prefix Tuning — Li & Liang (2021) [4].

WHAT IT IS
  The backbone is frozen. Every self-attention layer gets P learnable key vectors and P learnable value vectors
  (a "prefix") that all queries can attend to, in addition to the real tokens. Unlike prompt tuning (input layer
  only), this steers every layer. The paper reparameterises the prefix through a small MLP during training for
  stability (prefix = MLP(embedding)), which you may or may not do.

PLUMBING HOOK ALREADY IN PLACE (phase1_models/fastattn_ab.py and fastattn_cd.py)
  If an attention module has an attribute `prefix_kv`, the fused attention calls
        pk, pv = attn.prefix_kv(batch_size)        # each (B, n_kv_heads, P, d_head)
  prepends them to that layer's keys/values (BEFORE the GQA head repetition; no RoPE is applied to them) and extends
  the attention mask so every query sees all P prefix positions (causal / bidirectional / prefix-LM rules unchanged
  for the real tokens). Without `prefix_kv` attention is exactly as before. Supported: self-attention of A/B
  (fastattn_ab), C encoder/decoder self-attention and D prefix-LM attention (fastattn_cd); not cross-attention.
  Shapes: A has n_kv_heads = n_heads = 12; B/C/D have n_kv_heads = 4; d_head = 64 everywhere.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prefix_tuning", "n_prefix": ..., "reparam_hidden": ...}

SUGGESTED STRUCTURE
  class LayerPrefix(nn.Module):
      __init__(self, n_kv_heads, d_head, P, ...): self.k = nn.Parameter((n_kv_heads, P, d_head)), self.v = same
          (or an embedding + MLP producing them, for the reparameterised version)
      forward / __call__(self, B): return (self.k.expand(B, ...), self.v.expand(B, ...))
  def apply(backbone, n_prefix, ...):
      freeze(backbone)
      for block in backbone.blocks:                       # input -> output order; C "decoder" mode: enc + dec blocks
          attn = block.attn
          prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
          attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
          attn.prefix_kv = prefix                         # the hook the fused attention calls
  IMPORTANT: assign the module as a submodule (setattr on the attention module) so its parameters are part of
  model.parameters() / state_dict; `prefix_kv` must be callable with the batch size.
"""



import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401


class LayerPrefix(nn.Module):
  def __init__(self,n_kv_heads,d_head,P):
    super().__init__()
    self.n_kv_heads = n_kv_heads
    self.d_head = d_head
    self.P=P
    self.k = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    self.v = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    nn.init.normal_(self.k,mean=0,std=0.02)
    nn.init.normal_(self.v,mean=0,std=0.02)
  def forward(self,B):
    return (self.k.unsqueeze(0).expand(B,-1,-1,-1),self.v.unsqueeze(0).expand(B,-1,-1,-1))







def apply(backbone,n_prefix):
  freeze(backbone)
  for block in backbone.blocks:
    attn=block.attn
    prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
    attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
    attn.prefix_kv = prefix                         # the hook the fused attention calls

    

In [ ]:
%%writefile /kaggle/working/phase7/techniques/prompt_tuning.py
"""YOUR FILE: Prompt Tuning — Lester, Al-Rfou & Constant (2021) [5].

WHAT IT IS
  The whole backbone is frozen. P learnable "soft prompt" vectors (each of size d_model) are prepended to the
  input embeddings of every window; only those P x d_model numbers (plus the head) train.

PLUMBING HOOK ALREADY IN PLACE (backbones.Backbone)
  * backbone.n_virtual_tokens: set it to P in apply(). From then on the backbone itself prepends P placeholder ids
    (<pad>, id 0) to every window and removes the first P output positions again, so pooling / heads are unchanged.
    The plumbing checks window_len + P <= max_seq_len.
  * backbone.get_input_embedding() / backbone.set_input_embedding(module): the token embedding the prompt goes into
    (A/B/D: lm.tok_emb; C: the encoder's lm.tok_emb1).
  Your job: make positions [:, :P] of the embedding output learnable.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prompt_tuning", "n_tokens": ..., "init": ...}
      -> def apply(backbone, n_tokens, init=...):

SUGGESTED STRUCTURE
  class SoftPromptEmbedding(nn.Module):
      __init__(self, base_emb: nn.Embedding, n_tokens, init):
          keep base_emb (frozen); self.prompt = nn.Parameter(tensor of shape (n_tokens, d_model))
          init options from the paper: random (small normal) or copied from the embeddings of real vocabulary tokens
      forward(self, ids):  e = base_emb(ids)                     # (B, P + T, d), first P ids are placeholders
                           return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, P:]], dim=1)
  def apply(backbone, n_tokens, init):
      freeze(backbone)
      backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
      backbone.n_virtual_tokens = n_tokens
  Note: A/B/D tie the LM head to tok_emb; the head is not used in Phase 3 (replaced by Identity), so wrapping is safe.
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401

class SoftPromptEmbedding(nn.Module):
  def __init__(self,base_emb:nn.Embedding,n_tokens,init):
      super().__init__()
      self.base_emb = base_emb
      self.d_model = base_emb.embedding_dim
      self.n_tokens = n_tokens
     
      self.prompt = nn.Parameter(torch.empty(self.n_tokens, self.d_model))

      if init == "random":
            nn.init.normal_(self.prompt, mean=0.0, std=0.02)

      else:
            raise ValueError(f"Unknown prompt initialization: {init}")



  def forward(self,ids):
    e = self.base_emb(ids)
    B,_,_ = e.shape
    return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, self.n_tokens:]], dim=1)
def apply(backbone,n_tokens,init):
  freeze(backbone)
  backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
  backbone.n_virtual_tokens = n_tokens


    

In [ ]:
%%writefile /kaggle/working/phase7/techniques/ia3.py
"""YOUR FILE: IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations", Liu et al. (2022) [13].

WHAT IT IS
  The backbone is frozen; the only new parameters are three learned VECTORS per block that rescale activations
  element-wise (no matrices, so very few parameters):
      keys:    k' = l_k ⊙ k        l_k ∈ R^{dim of k_proj output}
      values:  v' = l_v ⊙ v        l_v ∈ R^{dim of v_proj output}
      FFN:     h' = l_ff ⊙ act(W1 x)   applied to the FFN's inner activation, i.e. to the INPUT of the second FFN
                                        linear (ffn.net.2), l_ff ∈ R^{d_ff}
  All vectors are initialised to ONES, so at the start the model is exactly the pretrained one.
  (Note: in your models k_proj / v_proj output n_kv_heads * d_head — 256 for B/C/D, 768 for A — use out_features.)

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None      (same contract as your other techniques)
      e.g. config {"name": "ia3"} (no options), or your own options such as which blocks / which of k, v, ffn.
      On exit only the l_k / l_v / l_ff vectors (plus nothing else) should have requires_grad=True.

SUGGESTED STRUCTURE
  class ScaleOutput(nn.Module):  wraps an nn.Linear; forward(x) = base(x) * self.scale   (scale = ones(out_features))
      -> use for attn.k_proj and attn.v_proj via replace_linears
  class ScaleInput(nn.Module):   wraps an nn.Linear; forward(x) = base(x * self.scale)   (scale = ones(in_features))
      -> use for ffn.net.2 (its input IS act(W1 x)), via replace_linears
  def apply(backbone, **cfg):
      freeze(backbone); replace_linears(... k_proj / v_proj -> ScaleOutput ...); replace_linears(... "ffn.net.2" -> ScaleInput ...)
  Parameter count check: per block len(l_k) + len(l_v) + d_ff (B: 256 + 256 + 3456 = 3,968 per block).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401

class ScaledOutput(nn.Module):
    def __init__(self,base:nn.Linear):
        super().__init__()
        self.base = base
        self.out_features = base.out_features
        self.in_features = base.in_features
        self.scale = nn.Parameter(torch.ones(self.out_features))
    def forward(self,x:torch.Tensor):
        x = self.base(x)
        return x*self.scale

class ScaledInput(nn.Module):
    def __init__(self,base:nn.Linear):
        super().__init__()
        self.base = base
        self.out_features = base.out_features
        self.in_features = base.in_features
        self.scale = nn.Parameter(torch.ones(self.in_features))
    def forward(self,x:torch.Tensor):
        x = self.base(self.scale*x)
        return x



def apply(backbone):
    freeze(backbone)

    replace_linears(
        backbone,
        lambda name, lin: name.endswith(("k_proj", "v_proj", "ffn.net.2")),
        lambda name, lin: (
            ScaledOutput(lin)
            if name.endswith(("k_proj", "v_proj"))
            else ScaledInput(lin)
        )
    )

In [ ]:
%%writefile /kaggle/working/phase7/techniques/kd.py
"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2] -- COMPULSORY experiment.

Used when the config has a "kd" block: {"teacher_run": <run dir of a finished teacher>, ...your options}.
The plumbing loads the teacher (frozen, eval mode), runs it on exactly the same windows as the student and calls

kd_loss(student_logits, teacher_logits, tags, **cfg) -> scalar
    student_logits, teacher_logits: (B, T, n_tags) float32 (same tag set: checked)
    tags: (B, T) long gold tags, -100 = ignore
    cfg: the "kd" dict minus "teacher_run" (e.g. your temperature / mixing weight)
This loss REPLACES the normal task loss for the student, so include the gold-label term yourself if you want it.
The student's own adaptation is chosen as usual with "technique" (e.g. full_ft on a smaller backbone).
"""

"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2]."""

import torch
import torch.nn.functional as F


def kd_loss(student_logits, teacher_logits, tags, **cfg):
    """
    Knowledge-distillation loss for token classification / NER.

    Args:
        student_logits: (B, T, C) student logits.
        teacher_logits: (B, T, C) teacher logits.
        tags: (B, T) gold tag IDs; -100 means ignore.
        cfg:
            temperature: softmax temperature (default 2.0).
            alpha: weight for KD loss (default 0.5).
                   Total loss = alpha * KD + (1-alpha) * CE.

    Returns:
        Scalar loss.
    """
    temperature = cfg.get("temperature", 2.0)
    alpha = cfg.get("alpha", 0.5)

    if temperature <= 0:
        raise ValueError("temperature must be positive")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be between 0 and 1")

    if student_logits.shape != teacher_logits.shape:
        raise ValueError(
            "Student and teacher logits must have the same shape"
        )

    mask = tags != -100

  
    if not mask.any():
        return student_logits.sum() * 0.0

    
    teacher_logits = teacher_logits.detach()

  
    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits / temperature, dim=-1
    )

    
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none"
    ).sum(dim=-1)

    kd_term = kl_per_token[mask].mean() * (temperature ** 2)

   
    ce_term = F.cross_entropy(
        student_logits[mask],
        tags[mask],
        weight=cfg.get("weight"),
    )

   
    return alpha * kd_term + (1.0 - alpha) * ce_term

In [ ]:
%%writefile /kaggle/working/phase7/experiments.py
"""PCR experiment matrix: YOUR values (course rule 9), transcribed from your Phase-7 config (2026-11-06).
BASE = your COMMON block; pcr_config / matrix / SPLIT / KD are yours unchanged.

build(phase1, data_dir, tokenizer, out_dir, cache_dir) -> {run name: config dict}
Spec 13.2: >= 6 substantially different experiments, >= 5 different techniques from {full_ft, partial_ft, qlora,
lorma (compulsory), prefix_tuning, prompt_tuning, ia3}, + 1 KD run (counts as one of the 6).
Field meanings: train_pcr.py docstring.
"""
import copy

BASE = {
    "fast_attention": True,
    "backbone": {"prefix": None, "encdec_mode": None, "truncate_layers": None, "dropout": None},
    "chunking": {"strategy": "even", "max_chunks": 6, "chunk_len": 896, "stride": 640},
    "chunk_pool": "mean",
    "doc_pool": "attn",
    "proj_dim": 256,
    "similarity": "cosine",
    "temperature": 0.05,
    "negatives": {"source": "bm25", "per_positive": 2, "bm25_top": 100, "in_batch": True},
    "technique": {"name": None},
    "kd": None,
    "train": {},
}


def deep_update(d, u):
    for k, v in u.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict):
            deep_update(d[k], v)
        else:
            d[k] = copy.deepcopy(v)
    return d


def pcr_config(technique, lr, epochs=2, head_lr=3e-4, changes=None, tech_opts=None):
    c = {
        "technique": {"name": technique, **(copy.deepcopy(tech_opts) if tech_opts else {})},
        "train": {
            "epochs": epochs, "batch_pairs": 8, "micro_docs": 2, "lr": lr, "head_lr": head_lr,
            "weight_decay": 0.01, "warmup_frac": 0.05, "schedule": "cosine", "grad_clip": 1.0, "seed": 42,
            "log_every": 20, "infer_micro_docs": 4, "max_minutes": 250, "eval_every_steps": None, "patience": 1,
            "eval_at_start": True,
        },
    }
    if changes:
        deep_update(c, changes)
    return c


LORMA_OPTS = {"r": 16, "alpha": 32, "targets": ["attn", "ffn"]}
QLORA_OPTS = {"r": 16, "alpha": 32, "targets": ["attn", "ffn"], "block_size": 64}


def matrix():
    """-> {name: (model key, overrides)}; model keys: A, B2, B3, C, D."""
    return {
        # P1: Full fine-tuning
        "P1": ("B2", pcr_config("full_ft", lr=1e-5, epochs=2)),
        # P2: Partial fine-tuning
        "P2": ("B2", pcr_config("partial_ft", lr=2e-5, epochs=2, tech_opts={"k": 6})),
        # P3: QLoRA
        "P3": ("B2", pcr_config("qlora", lr=1e-4, epochs=2, tech_opts=QLORA_OPTS)),
        # P4: LoRMA - KD teacher
        "P4": ("B2", pcr_config("lorma", lr=5e-5, epochs=2, tech_opts=LORMA_OPTS)),
        # P5: Prefix tuning
        "P5": ("B2", pcr_config("prefix_tuning", lr=2e-4, epochs=2, tech_opts={"n_prefix": 32})),
        # P6: LoRMA + mean document pooling
        "P6": ("B2", pcr_config("lorma", lr=5e-5, epochs=2, tech_opts=LORMA_OPTS, changes={"doc_pool": "mean"})),
        # P7: LoRMA + random negatives
        "P7": ("B2", pcr_config("lorma", lr=5e-5, epochs=2, tech_opts=LORMA_OPTS,
                                changes={"negatives": {"source": "random", "per_positive": 2, "bm25_top": None,
                                                       "in_batch": True}})),
    }


SPLIT = {
    1: ["P1", "P3", "P5", "P7"],
    2: ["P2", "P4", "P6", "P8_kd"],
}

KD_NAME = "P8_kd"
KD_TEACHER = "P4"          # P4 is queued before P8_kd on account 2
KD_STUDENT = ("B2", pcr_config("full_ft", lr=2e-5, epochs=2,
                               changes={"backbone": {"truncate_layers": 9}, "kd": {"temperature": 2.0, "alpha": 0.5}}))


def make_cfg(name, model_key, overrides, phase1, data_dir, tokenizer, out_dir, cache_dir):
    assert model_key, f"{name}: set its model key (A / B2 / B3 / C / D) in experiments.py"
    c = deep_update(copy.deepcopy(BASE), overrides)
    c.update(name=name, phase1_ckpt=phase1[model_key], data_dir=data_dir, tokenizer=tokenizer, out_dir=out_dir,
             cache_dir=cache_dir, model_key=model_key)
    assert c["phase1_ckpt"], f"{name}: no checkpoint for model {model_key} (attach it)"
    return c


def build(phase1, data_dir, tokenizer, out_dir, cache_dir):
    return {n: make_cfg(n, m, o, phase1, data_dir, tokenizer, out_dir, cache_dir) for n, (m, o) in matrix().items()}


def build_kd(teacher_run, phase1, data_dir, tokenizer, out_dir, cache_dir, name=KD_NAME):
    m, o = copy.deepcopy(KD_STUDENT)
    o["kd"]["teacher_run"] = teacher_run
    return make_cfg(name, m, o, phase1, data_dir, tokenizer, out_dir, cache_dir)

## Stage 2 — data, schema check, Phase-1 inputs, token cache

In [ ]:
# Phase_7_PCR from the gated HF dataset (Kaggle secret "HF"), file schema check, Phase-1 inputs, token cache
from kaggle_secrets import UserSecretsClient
_hf, _errs = None, {}
for _label in ("HF", "HF_TOKEN"):
    try:
        _hf = UserSecretsClient().get_secret(_label)
        if _hf:
            print(f"Hugging Face token: Kaggle secret '{_label}'")
            break
    except Exception as e:
        _errs[_label] = type(e).__name__
if not _hf:
    raise SystemExit(f"Could not read a Kaggle secret 'HF' or 'HF_TOKEN' ({_errs}): Add-ons -> Secrets -> add your Hugging "
                     "Face token under one of these labels and TICK it for this notebook; then rerun this cell.")
subprocess.run([sys.executable, "fetch_data.py", "--out", os.path.join(WORK, "data")], check=True,
               env=dict(os.environ, HF_TOKEN=_hf.strip()))
del _hf
DATA_DIR = os.path.join(WORK, "data", "Phase_7_PCR")

import pcr_data
print(json.dumps(pcr_data.schema_report(DATA_DIR), indent=1))        # check the field names the loader picked up
SPLIT_DATA = {s: pcr_data.load_split(DATA_DIR, s) for s in ("train", "dev", "test")}
for s, (q, p) in SPLIT_DATA.items():
    print(s, pcr_data.split_stats(q, p))

def find_all(pattern):
    return sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

def find_one(pattern):
    hits = find_all(pattern)
    if len(hits) > 1:
        print(f"  note: {len(hits)} matches for {pattern}, using the first: {hits}")
    return hits[0] if hits else None

def sha8(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()[:8]

TOKS = find_all("legal_bpe.json")
assert TOKS, "legal_bpe.json not found under /kaggle/input (Add Input: your Phase-1 notebook outputs)"
tok_hashes = {t: sha8(t) for t in TOKS}
assert len(set(tok_hashes.values())) == 1, f"different legal_bpe.json files attached: {tok_hashes}"
TOKENIZER = TOKS[0]
print("tokenizer sha256:", tok_hashes[TOKENIZER], "(Phase-1 legal BPE was 38f51f14)")

PHASE1 = {"A": find_one("run_A/best.pt"), "B2": find_one("run_B2/best.pt"), "B3": find_one("ctxext/best_newctx.pt"),
          "C": find_one("run_C/best.pt"), "D": find_one("run_D/best.pt")}
for k, v in PHASE1.items():
    if v is None:
        print(f"Phase-1 {k}: not attached"); continue
    ck = torch.load(v, map_location="cpu", weights_only=False, mmap=True)
    c = ck["config"]
    lim = c["model"]["max_seq_len"] if "model" in c else c["encoder"]["max_seq_len"]
    print(f"Phase-1 {k}: {v}\n    tag={c.get('tag')} module={c['module']} step={ck.get('step')} max_len={lim}")
    del ck

from a1_tokenizer import ByteBPE
_tok = ByteBPE.from_json(TOKENIZER)
t0 = time.time()
for s, (q, p) in SPLIT_DATA.items():                 # same cache files train_pcr.load_data uses
    pcr_data.encode_docs(_tok, q, os.path.join(WORK, "cache", f"{s}_queries_ids.npz"))
    pcr_data.encode_docs(_tok, p, os.path.join(WORK, "cache", f"{s}_pool_ids.npz"))
print(f"token cache ready in {time.time() - t0:.0f} s")
del _tok

## Stage 2b — BM25 reference score on dev (not a submission)

In [ ]:
# BM25 reference on dev (lexical, no model; NOT a valid Phase-7 experiment: every run must start from a Phase-1 model).
# Useful as a yardstick for the dev scores below and for hard-negative quality.
subprocess.run([sys.executable, "pcr_select.py", "--bm25-only", "--data-dir", DATA_DIR,
                "--out", os.path.join(WORK, "ensembles")], check=True)

## Stage 3 — configuration (experiments.py)

In [ ]:
# Experiment matrix = experiments.py (YOUR values). RUN = what this session trains, in queue order.
import importlib, experiments, train_pcr, techniques
importlib.reload(experiments)
EXPERIMENTS = experiments.build(PHASE1, DATA_DIR, TOKENIZER, os.path.join(WORK, "runs"), os.path.join(WORK, "cache"))
ACCOUNT = None                                  # 1 / 2 (two Kaggle accounts in parallel) or "all"
assert ACCOUNT in (1, 2, "all"), "set ACCOUNT = 1, 2 or 'all' in this cell"
RUN = (list(EXPERIMENTS) + [experiments.KD_NAME]) if ACCOUNT == "all" else list(experiments.SPLIT[ACCOUNT])
unknown = [n for n in RUN if n not in EXPERIMENTS and n != experiments.KD_NAME]
assert not unknown, f"SPLIT names not in the matrix: {unknown}"

def find_run(name):
    # a finished run (summary.json + best_model.pt) in WORK/runs or in an attached earlier output of this notebook
    for d in [os.path.join(WORK, "runs", name)] + sorted(glob.glob(f"/kaggle/input/**/runs/{name}", recursive=True)):
        if all(os.path.exists(os.path.join(d, f)) for f in ("summary.json", "best_model.pt", "config.json")):
            if not json.load(open(os.path.join(d, "summary.json"))).get("smoke"):
                return d
    return None

if experiments.KD_NAME in RUN:                  # KD: teacher queued earlier in RUN (run_many waits) or finished/attached
    _t = experiments.KD_TEACHER
    assert _t, "set experiments.KD_TEACHER (a run of the matrix)"
    _td = os.path.join(WORK, "runs", _t) if _t in RUN[:RUN.index(experiments.KD_NAME)] else find_run(_t)
    assert _td, f"KD teacher {_t} is neither queued before {experiments.KD_NAME} nor a finished attached run"
    EXPERIMENTS[experiments.KD_NAME] = experiments.build_kd(_td, PHASE1, DATA_DIR, TOKENIZER, os.path.join(WORK, "runs"),
                                                            os.path.join(WORK, "cache"))

def not_implemented(tech_name):
    src = inspect.getsource(techniques.get(tech_name))
    return "raise NotImplementedError" in src

DONE = {n: find_run(n) for n in RUN if find_run(n)}
if DONE:
    print("already finished (reused, not trained again):", DONE)
CFG_PATHS = []
for name in RUN:
    if name in DONE:
        continue
    cfg = EXPERIMENTS[name]
    if not_implemented(cfg["technique"]["name"]):
        print(f"SKIP {name}: techniques/{cfg['technique']['name']}.py is not implemented"); continue
    train_pcr.check_config(cfg)
    path = os.path.join(WORK, "configs", f"{name}.json")
    json.dump(cfg, open(path, "w"), indent=2)
    CFG_PATHS.append(path)
    ng, ch = cfg["negatives"], cfg["chunking"]
    print(f"{name:12s} | model={cfg['model_key']:3s} | technique={cfg['technique']['name']:13s} | kd={bool(cfg.get('kd'))} | "
          f"lr={cfg['train']['lr']} | {cfg['similarity']}/T={cfg['temperature']} | neg={ng['source']}x{ng['per_positive']} "
          f"in_batch={ng['in_batch']} | chunks={ch['strategy']} x{ch['max_chunks']} | pool={cfg['chunk_pool']}/{cfg['doc_pool']}")
print(f"{len(CFG_PATHS)} runs queued")

## Stage 3b — real-model smoke test (memory, speed, time per run)

In [ ]:
# Real-model smoke test on both GPUs BEFORE the long runs: every queued config unchanged (checkpoint, technique, chunking,
# micro-batch), on the 8 longest train queries + small pools, 4 optimizer steps, dev/test on 4 queries, zip. Measures peak
# GPU memory and training speed -> rough minutes per full run (written into the configs as est_minutes, used by the
# scheduler when train.max_minutes is unset). Outputs: runs_smoke/.
import numpy as np
RUN_SMOKE = True
def _tokens(split, part):
    z = np.load(os.path.join(WORK, "cache", f"{split}_{part}_ids.npz"))
    return np.diff(z["offsets"])
if RUN_SMOKE and CFG_PATHS:
    smoke_paths = []
    for path in CFG_PATHS:
        c = json.load(open(path))
        c["smoke"] = {"steps": 4, "queries": 8}
        c["out_dir"] = os.path.join(WORK, "runs_smoke")
        if c.get("kd") and os.path.basename(c["kd"]["teacher_run"].rstrip("/")) in RUN:     # the teacher's smoke run
            c["kd"]["teacher_run"] = os.path.join(WORK, "runs_smoke", os.path.basename(c["kd"]["teacher_run"].rstrip("/")))
        sp = os.path.join(WORK, "configs", f"smoke_{c['name']}.json")
        json.dump(c, open(sp, "w"), indent=2)
        smoke_paths.append(sp)
    t0 = time.time()
    gpus = ",".join(str(i) for i in range(max(1, N_GPUS)))
    sp_proc = subprocess.run([sys.executable, "run_many.py", "--gpus", gpus, "--every", "60", *smoke_paths])
    print(f"\nsmoke test: {(time.time() - t0) / 60:.1f} min")
    n_pairs = sum(len(q["relevant"]) for q in SPLIT_DATA["train"][0])
    for path in CFG_PATHS:
        c = json.load(open(path))
        sm = os.path.join(WORK, "runs_smoke", c["name"], "summary.json")
        if not os.path.exists(sm):
            print(f"  FAIL {c['name']:12s} -> see {os.path.join(WORK, 'runs_smoke', c['name'] + '.stdout')}"); continue
        s_ = json.load(open(sm))
        tok_s = max(1, s_["train_tok_s"])
        L, k = s_["chunking"]["chunk_len"], c["chunking"]["max_chunks"]
        per_doc = lambda a: float(np.mean(np.minimum(a, L * k)))
        docs_per_pair = 2 + c["negatives"]["per_positive"]
        train_tok = n_pairs * docs_per_pair * per_doc(_tokens("train", "pool"))        # tokens re-encoded with grad
        eval_tok = per_doc(_tokens("dev", "pool")) * len(SPLIT_DATA["dev"][1]) + per_doc(_tokens("dev", "queries")) * len(SPLIT_DATA["dev"][0])
        n_eval = c["train"]["epochs"] * (1 + (n_pairs // c["train"]["batch_pairs"]) // (c["train"]["eval_every_steps"] or 10**9)) + 2
        est = 1.3 * (c["train"]["epochs"] * train_tok / tok_s + n_eval * eval_tok / (2.5 * tok_s)) / 60 + 15
        c["est_minutes"] = round(est, 1)
        json.dump(c, open(path, "w"), indent=2)
        print(f"  OK   {c['name']:12s} peak {s_['peak_mem_GB']} GB of ~15 | trainable {s_['params']['trainable']:,} "
              f"({s_['params']['trainable_pct']}%) | {tok_s} tok/s | ~{c['est_minutes']} min per full run (rough)")
    assert sp_proc.returncode == 0, "smoke test failed: fix the runs marked FAIL before launching the full runs"
    for ck in glob.glob(os.path.join(WORK, "runs_smoke", "*", "best_model.pt")):
        os.remove(ck)
    print("SMOKE TEST PASSED -> run the next cell for the full runs")

## Stage 4 — main runs incl. KD (both GPUs)

In [ ]:
# Main runs on BOTH T4s (one run per GPU at a time, shared queue in RUN order; the KD run waits for its teacher).
# A run is only STARTED if its time (train.max_minutes, else est_minutes) fits before SESSION_HOURS; others are SKIPPED.
assert N_GPUS >= 1, "no GPU visible: set Accelerator to 'GPU T4 x2'"
gpus = ",".join(str(i) for i in range(N_GPUS))
SESSION_HOURS = 11.5
STOP_AT = SESSION_START + SESSION_HOURS * 3600
print(f"{(STOP_AT - time.time()) / 3600:.2f} h of session left for starting runs")
if CFG_PATHS:
    proc = subprocess.run([sys.executable, "run_many.py", "--gpus", gpus, "--every", "600", "--stop-at", str(STOP_AT),
                           "--default-minutes", "180", "--margin", "15", *CFG_PATHS])
    print(f"run_many exit code {proc.returncode} | session time used {(time.time() - SESSION_START) / 3600:.2f} h")

## Stage 5 — results and zips

In [ ]:
# results table (one row per finished run, facts only) + all zips in /kaggle/working/submissions
import pandas as pd
rows = []
os.makedirs("/kaggle/working/submissions", exist_ok=True)
for n in list(experiments.matrix()) + [experiments.KD_NAME]:
    d = find_run(n)
    if not d:
        rows.append(dict(name=n, status="not finished")); continue
    s = json.load(open(os.path.join(d, "summary.json")))
    a7 = s["dev"]["at7"]
    rows.append(dict(name=n, status="ok" if s["full_epoch_completed"] else "INCOMPLETE EPOCH", model=s["model_key"],
                     technique=s["technique"]["name"], kd=s["kd"], sim=s["similarity"], T=s["temperature"],
                     neg=f"{s['negatives']['source']}x{s['negatives']['per_positive']}" + ("+inbatch" if s["negatives"]["in_batch"] else ""),
                     chunks=f"{s['chunking']['strategy']}x{s['chunking']['max_chunks']}@{s['chunking']['chunk_len']}",
                     pool=f"{s['chunk_pool']}/{s['doc_pool']}", params=s["params"]["total"],
                     trainable=s["params"]["trainable"], trainable_pct=s["params"]["trainable_pct"],
                     dev_F1_at7_untrained=s["start_dev_f1_at7"], dev_F1_at7=a7["micro_f1"], dev_P_at7=a7["micro_p"],
                     dev_R_at7=a7["micro_r"], dev_R_at100=s["dev"]["recall@100"], selected_at=s["selected_at"],
                     epochs=s["epochs_completed"], train_min=round(s["train_seconds"] / 60, 1), peak_GB=s["peak_mem_GB"],
                     infer_ms_per_doc=s["infer_ms_per_doc"]))
    z = os.path.join(d, f"submission_{n}.zip")
    if os.path.exists(z):
        shutil.copy(z, "/kaggle/working/submissions/")
df = pd.DataFrame(rows)
display(df)
df.to_csv("/kaggle/working/submissions/summary.csv", index=False)
print(sorted(os.listdir("/kaggle/working/submissions")))

## Stage 6 — score fusion (your weights)

In [ ]:
# Score fusion of finished runs (+ optional BM25), YOUR weights (nothing is tuned): per-query normalised scores
# (norm: zscore | minmax | none), weighted sum -> dev Micro-F1@7 printed, test submission written.
#   name -> {"members": [run names], "weights": [one per member], "bm25_weight": float, "norm": str}
FUSION = {
    "ENS1": {"members": None, "weights": None, "bm25_weight": None, "norm": None},
}
for fname, e in FUSION.items():
    if not e["members"]:
        print(f"{fname}: skipped (fill members / weights / bm25_weight / norm)"); continue
    dirs = [find_run(m) for m in e["members"]]
    assert all(dirs), f"{fname}: unfinished members {[m for m, d in zip(e['members'], dirs) if not d]}"
    subprocess.run([sys.executable, "pcr_select.py", "--name", fname, "--out", os.path.join(WORK, "ensembles"),
                    "--data-dir", DATA_DIR, "--members", *dirs, "--weights", *map(str, e["weights"]),
                    "--bm25-weight", str(e["bm25_weight"]), "--norm", e["norm"]], check=True)
    shutil.copy(os.path.join(WORK, "ensembles", fname, f"submission_{fname}.zip"), "/kaggle/working/submissions/")